## IMPORTATION DES BIBLIOTHEQUES ET PARAMETRES GLOBAUX

In [11]:
### Importation des bibliothèques standards et paramètres globaux ###
import os
import json
import importlib
import time
from datetime import datetime
from pathlib import Path; import ipynbname; 

# --- Bibliothèques scientifiques ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

# --- Configuration de l'affichage matplotlib ---
%matplotlib qt 
# Mode interactif pour les figures
base_path = ipynbname.path().parent.parent

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.size': 20
})

# --- Options d'affichage Pandas ---
pd.set_option('display.max_rows', 100)
pd.options.display.float_format = '{:,.2f}'.format

#pd.options.display.float_format = '{:,.2f}'.format  # Affichage avec séparateur de milliers

# ==============================
# PARAMÈTRES GLOBAUX
# ==============================

# Obtenir la date du jour au format jjmmaaa
date_du_jour = datetime.now().strftime("%d%m%y")


### Importation des fonctions définies dans les fichiers biodiv

In [12]:

import affichage_carte_biodiv
importlib.reload(affichage_carte_biodiv)
from affichage_carte_biodiv import (
    configurer_carte,
    ajouter_couche_SIG,
    ajouter_couche_continue,
    ajouter_couche_discrete,
    ajouter_couche_statut,
    ajouter_couche_point,
    afficher_carte_defaut,
    afficher_fond_carte,
    afficher_carte_interactive
)
import normalisation_biodiv
importlib.reload(normalisation_biodiv)
from normalisation_biodiv import (
    normaliser_par_maille,
    normaliser_par_maille_et_clade,
    normaliser_par_aire,
    normaliser_par_aire_et_clade,
    normaliser_par_espece,
    normaliser_par_clade,
    normaliser_unique,
    normaliser_log,
    normaliser_par_periode
)
import exploration_biodiv
importlib.reload(exploration_biodiv)
from exploration_biodiv import (
    filtrer_top_global,
    filtrer_top_mailles,
    filtrer_top,
    afficher_top_especes,
    chercher_espece,
    explorer_clade,
    chercher_especes_protegees
)
import correlation_prediction
importlib.reload(correlation_prediction)
from correlation_prediction import (
    calculer_matrice_correlation,
    calculer_correlation_sujet,
    recalculer_nombreObs_par_correlation,
    calculer_prediction,
    calculer_seuil,
    recherche_espece_absente,
    prepare_data,
    fit_and_plot,
    plot_residuals,
    appliquer_transformation
)
import biodiversite_endemisme_biodiv
importlib.reload(biodiversite_endemisme_biodiv)
from biodiversite_endemisme_biodiv import (
    calculer_shannon,
    calculer_simpson,
    calculer_WE,
    calculer_indices,
    calculer_entropie_quadratique_taxo,
    calcul_indices_par_zone,
    comparer_indices
)
import clustering_geo_biodiv
importlib.reload(clustering_geo_biodiv)
from clustering_geo_biodiv import (
    analyser_composantes_principales,
    former_cluster_biogeo,
    determiner_k,
    etudier_composition_cluster,
    kmeans_with_spatial_constraint,
    compute_spatial_centroids,
    assign_missing_clusters,
    former_cluster_biogeo_avec_critere_spatial,
    calculer_inertie
)

import clustering_espece_biodiv
importlib.reload(clustering_espece_biodiv)
from clustering_espece_biodiv import (
    generer_dendogram,
    former_cluster_espece,
    chercher_numcluster_espece,
    lister_especes_dans_cluster,
    grouper_par_cluster,
    etudier_un_cluster_local,
    rechercher_especes_localement_absentes,
    creer_pivot,
    clustering_hierarchique,
    clustering_fuzzy,
    clustering_gmm,
    clustering_reseau,
    clustering_lda,
    former_communautes_all
)


import fonctions_annexes_biodiv
importlib.reload(fonctions_annexes_biodiv)
from fonctions_annexes_biodiv import (
    round_to_sig,
    generer_dictionnaire_taxonomie,
    afficher_dataframe,
    completer_df,
    lister_mailles_dans_site,
    ajouter_nom_site_df,
    filtrer_grille,
    afficher_carte_monde,
    filtrer_geo_coord,
    filtrer_categorie,
    filtrer_geographie_zone
)

import evolution_temporelle
importlib.reload(evolution_temporelle)
from evolution_temporelle import (
    suivre_disparition_geo,
    determiner_statut
)

## IMPORTATION DES DONNEES

In [13]:
# Définition des régions par continent
regions = {
    "Amerique_centrale": ['Panama', 'Costa Rica', 'Nicaragua', 'El Salvador', 'Honduras', 
                          'Guatemala', 'Belize', 'Colombia', 'Mexico', 'Cuba'],
    "Amerique_Sud": ['Colombia', 'Brazil', 'Argentina', 'Peru', 'Ecuador', 'Chile', 
                     'Bolivia', 'Paraguay', 'Uruguay', 'Venezuela'],
    "Asie_du_sud": ['India', 'Sri Lanka', 'Bangladesh', 'Nepal', 'Bhutan'],
    "Asie_de_est": ['Japan', 'Republic of Korea'],
    "Asie_du_sud_est": ["Laos", "Myanmar", "Thailand", 
                        "Malaysia", "Vietnam", "Cambodia", "Philippines"],
    "Europe": ["Spain", "Italy", 'Slovenia', 'Switzerland', 'Croatia', 'Portugal', "Austria", 
               "France", "Germany", "Greece", "Slovakia", "Czechia", "Poland", "Lithuania", 
               "Latvia", "Estonia", "Finland", "Hungary", 'Romania', 'Ukraine', 'Bulgaria',
               'North Macedonia', 'Bosnia and Herzegovina', 'Serbia', 'Kosovo', 'Albania', 
               'Moldova', 'Turkey', 'Belarus', 'Montenegro', 'Cyprus'],
    "Med": ["Spain", "France", 'Italy', 'Slovenia', 'Croatia', 'Greece', "Turkey", 
            "Tunisia", "Algeria", "Morocco"],
    "Maghreb": ["Tunisia", "Algeria", "Morocco"],
    "East_africa": ['Ethiopia', 'Kenya', 'Uganda', 'Tanzania', 'Rwanda', 'Burundi', 'South Sudan'],
    "Afrique": ["Tunisia", "Algeria", "Morocco","Gabon"]
    
}
# Définition des filtres sous forme de dictionnaire
filtres = {
    "arbres": {"class": ["Pinopsida"], "order": ["Fagales"], "genus": ["Crataegus", "Prunus", "Malus", "Sorbus", "Pyrus"]},
    "arbres_reduit": {"class": ["Pinopsida"], "order": ["Fagales"]},
    "plantes": {"kingdom": ["Plantae"]},
     "plantes": {"phylum": ["Tracheophyta"]},
    "mammifères": {"class": ["Mammalia"]},
    "poissons": {"class": ["Actinopterygii", "Myxini", "Leptocardii", "Holocephali", "Dipneusti", "Petromyzonti", "Elasmobranchii"],
                 "order": ["Perciformes", "Tetraodontiformes", "Clupeiformes", "Characiformes", "Syngnathiformes", "Beloniformes", "Gymnotiformes"]},
    "oiseaux": {"class": ["Aves"]},
    "reptiles": {"class": ["Chelonii", "Squamata", "Crocodylia"]},
    "arthropodes": {"class": ["Insecta", "Arachnida", "Malacostraca", "Branchiopoda", "Chilopoda", "Diplopoda"]},
    "pollinisateurs": {"order": ["Hymenoptera", "Lepidoptera", "Diptera"]},
    "insectes": {"class": ["Insecta"]},
    "graminées": {"family": ["Poaceae"]},
    "mousses": {"order": ["Sphagnales", "Hypnales", "Dicranales", "Bryales", "Buxbaumiales", "Diphysciales", "Grimmiales",
                          "Andreaeales", "Polytrichales", "Hookeriales"]},
    "amphibiens": {"class": ["Amphibia"]},
    "mollusques": {"class": ["Gastropoda", "Bivalvia", "Cephalopoda", "Monoplacophora", "Scaphopoda"]},
    "araignées": {"order": ["Araneae"]},
    "champignons": {"kingdom": ["Fungi"]},
    "orchidées": {"family": ["Orchidaceae"]},
    "papillons": {"order": ["Lepidoptera"]}
}
regions_france= {
    "Sud Med": ["66", "11", "81", "34", "12", "48", "30", "07", "26", "84", "13", "05", "04", "83", "06"],
    "Pyrénées": ["09","11", "31", "32", "64", "65", "66"],
    "Hérault": ["34"],
    "Cévennes": ["12","30","34","48"],
    "Bouches-du-Rhône": ["13"]

}

### Importation des données

In [14]:
import os
import json
import pandas as pd
import geopandas as gpd

# ==============================
# CONFIGURATION
# ==============================
mode_importation = "pays"  # "pays", "local", ou "monde"
countries = ["France"]  # utilisé uniquement si mode_importation=="pays"
zone_name_short = "Sud Med"
ecosysteme = "terrestre"  # terrestre, maritime, combined
source = "GBIF"
cle_ID = "speciesKey"
grid_size_km = 2
fusion=True
seuil_fusion=5000
cle_geo = f"codeMaille{grid_size_km}Km"
bornes_temporelles = [1800, 1990, 2010, 2024]
chaine_bornes = "_".join(map(str, bornes_temporelles))

# Définition du nom de la zone
if mode_importation == "local":
    zones_locales = ["66", "11", "81", "34", "12", "48", "30", "07", "26", "84", "13", "05", "04", "83", "06"]
    zone_name = "_".join(countries + zones_locales)
elif mode_importation == "pays":
    zone_name = "_".join(countries)
else:
    zone_name = "Monde"

print(f"Mode choisi : {mode_importation} // nom de la zone : {zone_name_short} // ecosysteme : {ecosysteme} // nom du fichier : {ecosysteme}")

# ==============================
# CHEMINS
# ==============================
#base_path = r'C:\Users\Aubin\Documents\MANTIS'
data_path = os.path.join(base_path, 'Data')
save_path = os.path.join(base_path, 'Resultats', f'results_{date_du_jour}')
os.makedirs(save_path, exist_ok=True)
print(f"Répertoire {'existant' if os.path.exists(save_path) else 'créé'} : {save_path}")
sig_path = os.path.join(base_path, 'SIG')

# ==============================
# INITIALISATION DES DATAFRAMES
# ==============================
df_raw_import = pd.DataFrame()
carte_maille_raw = pd.DataFrame()
border_local_geo = gpd.GeoDataFrame()
eez_local_geo = gpd.GeoDataFrame()

# ==============================
# IMPORTATION DES DONNÉES GÉOGRAPHIQUES GLOBALES
# ==============================
border_geo = gpd.read_file(os.path.join(sig_path, "SIG_Global", "geoBoundariesCGAZ_ADM0.shp"))
eez_geo = gpd.read_file(os.path.join(sig_path, "SIG_Global", "eez_v11.gpkg"))
chemin_dico_cartes = os.path.join(sig_path, "SIG_Global", "dictionnaire_cartes.json")

with open(chemin_dico_cartes, "r", encoding="utf-8") as fichier:
    dictionnaire_cartes = json.load(fichier)

print("✅ Importation des données géographiques globales")

# ==============================
# IMPORTATION DES DONNÉES
# ==============================
# Définir la liste des pays/dossiers à importer
if mode_importation == "monde":
    path_source = os.path.join(data_path, source)
    countries_to_import = [
        d for d in os.listdir(path_source)
        if os.path.isdir(os.path.join(path_source, d)) and d.startswith(f"{source}_")
    ]
else:
    countries_to_import = countries

# Boucle sur chaque pays/dossier
for country in countries_to_import:
    country_safe = country  # nom du dossier, ex : "iNat_France"
    country = country.replace(f"{source}_", "")  # nom du pays, ex : "France"
    path_data_country = os.path.join(data_path, source,'processed', f"{source}_{country}")

    print(f"📥 Importation des données pour {country}...")

    # Chargement des frontières et EEZ uniquement pour pays/local
    if mode_importation != "monde":
        name_attribute = "shapeName"
        code_attribute = "shapeGroup"
        country_code = border_geo.loc[border_geo[name_attribute] == country, code_attribute].iloc[0]        
        border_local_geo = pd.concat([border_local_geo, border_geo[border_geo[code_attribute] == country_code]], ignore_index=True)
        eez_local_geo = pd.concat([eez_local_geo, eez_geo[eez_geo["ISO_SOV1"] == country_code]], ignore_index=True)

    # Chargement des mailles
    fichier_grille_local = os.path.join(path_data_country, f"grid_{country}_{ecosysteme}_{cle_geo}.geojson")
    if fusion:
        fichier_grille_local = os.path.join(path_data_country, f"grid_{country}_{ecosysteme}_{cle_geo}Adapt{seuil_fusion}.geojson")
    if os.path.exists(fichier_grille_local):
        carte_maille_raw = pd.concat([carte_maille_raw, gpd.read_file(fichier_grille_local)], ignore_index=True)
    else:
        print(f"⚠️ Fichier manquant : {fichier_grille_local}")

    # Chargement des données GBIF/iNat
    fichier_gbif = os.path.join(f"data_{source}_{country}_{ecosysteme}_{cle_geo}_{cle_ID}_periodes{chaine_bornes}.csv")
    if fusion:
        fichier_gbif = os.path.join(path_data_country, f"data_{source}_{country}_{ecosysteme}_{cle_geo}Adapt{seuil_fusion}_{cle_ID}_periodes{chaine_bornes}.csv")
    if os.path.exists(fichier_gbif):
        df_temp = pd.read_csv(fichier_gbif, dtype={"nombreObs": int, cle_ID: int})
        df_raw_import = pd.concat([df_raw_import, df_temp], ignore_index=True)
    else:
        print(f"⚠️ Fichier manquant : {fichier_gbif}")

print("✅ Importation terminée pour tous les pays/zones.")

# ==============================
# TRAITEMENT DES DONNÉES
# ==============================
# Filtrage des mailles valides
df_raw_import = df_raw_import[df_raw_import[cle_geo].isin(carte_maille_raw[cle_geo])]
carte_maille=carte_maille_raw.copy()
print("✅ Filtrage des mailles valides")

# Génération du dictionnaire taxonomique
dico_taxo = generer_dictionnaire_taxonomie(df_raw_import, cle_ID)
liste_col_taxo = dico_taxo.columns.tolist()

# Agrégation par période et clé
df_raw_import = df_raw_import.groupby([cle_geo, cle_ID, "periode"], observed=True)["nombreObs"].sum().reset_index()

# Fusion avec la taxonomie
df_raw_import = df_raw_import.merge(dico_taxo, on=cle_ID, how="left")
print("✅ Génération du dictionnaire taxonomique")

print("✅ Importation et traitement terminés")


Mode choisi : pays // nom de la zone : Sud Med // ecosysteme : terrestre // nom du fichier : terrestre
Répertoire existant : C:\Users\Aubin\Documents\MANTIS\Resultats\results_221025
✅ Importation des données géographiques globales
📥 Importation des données pour France...
✅ Importation terminée pour tous les pays/zones.
✅ Filtrage des mailles valides
✅ Génération du dictionnaire taxonomique
✅ Importation et traitement terminés


### Filtrage des données

In [15]:
# Filtrage des données et statistiques

# Filtrage géographique si nécessaire

filtrage_geo = False
carte_maille_saved=carte_maille.copy()
if filtrage_geo:
    zone_name_short="Sicily"
    afficher_carte_monde(border_geo, 15)
    lon_min, lon_max = 11, 15.65
    lat_min, lat_max = 36.6, 39
    filtrage_geo = input("Souhaitez-vous filtrer la carte géographiquement ? (o/n) ").lower() == 'o'
    carte_maille= filtrer_grille(carte_maille_saved, lat_min, lat_max, lon_min, lon_max)
    df_raw_import = df_raw_import[df_raw_import[cle_geo].isin(carte_maille[cle_geo])].copy()
# Afficher le nombre de mailles 
print(f"Nombre de mailles dans la zone selectionnées : {carte_maille[cle_geo].nunique()}")
print(f"\nNombre de mailles avec données : {df_raw_import[cle_geo].nunique()}")
print(f"\nNombre totale d'observation : {df_raw_import['nombreObs'].sum()}")
print("\n")


# Afficher les règnes existants
regnes_disponibles = df_raw_import['kingdom'].unique()
print("Règnes disponibles :", regnes_disponibles)


# Définir les règnes à inclure (True pour inclure, False pour exclure)
selection_regnes = {
    'Animalia': True,
    'Plantae': True,
    'Fungi': False,
    'Protozoa': False,
    'Chromista': False,
    'Bacteria': False,
    'Viruses': False,
    'Archaea': False# Modifier ici selon les besoins
    }

# Filtrer les données en fonction des règnes sélectionnés
regnes_a_garder = [regne for regne, garder in selection_regnes.items() if garder]
df_raw_filtred = df_raw_import[df_raw_import['kingdom'].isin(regnes_a_garder)].copy()
# Affichage des règnes filtrés
print("Règnes sélectionnés :", regnes_a_garder)
embranchements_disponibles = df_raw_filtred['phylum'].unique()
print("\n")
print("Embranchements disponibles :", embranchements_disponibles)
# Compter le nombre d'especes
df_raw_filtred = normaliser_unique(df_raw_filtred)  

# Calcul des statistiques
stats = df_raw_filtred.groupby(cle_geo)[['nombreObs', 'nombreObs_unique']].sum()

# Renommer la colonne 'nombreObs_unique' en 'nombreEspeces'
stats.rename(columns={'nombreObs': 'nombreObs par maille'}, inplace=True)
stats.rename(columns={'nombreObs_unique': 'nombreEspeces par maille'}, inplace=True)

# Création du tableau de synthèse
stats_summary = pd.DataFrame({
    "Moyenne": stats.mean(),
    "Médiane": stats.median(),
    "Minimum": stats.min(),
    "Maximum": stats.max(),
    "1er décile": stats.quantile(0.1),
    "9e décile": stats.quantile(0.9)
}).astype(int)
print("\n")
# Afprint(stats_summary)fichage sous forme de tableau
print(stats_summary)

# Copie des données avec périodes
df_biodiv_periode = df_raw_filtred.copy()

# Création du DataFrame sans période (somme globale)
df_biodiv_sansperiode = df_biodiv_periode.groupby([cle_geo, cle_ID], as_index=False)["nombreObs"].sum()
df_biodiv_sansperiode = df_biodiv_sansperiode.merge(dico_taxo, on=cle_ID, how="left")


Nombre de mailles dans la zone selectionnées : 15523

Nombre de mailles avec données : 15523

Nombre totale d'observation : 176625890


Règnes disponibles : ['Animalia' 'Fungi' 'Plantae' 'Chromista' 'Bacteria' 'Protozoa' 'Viruses'
 'Archaea' 'incertae sedis']
Règnes sélectionnés : ['Animalia', 'Plantae']


Embranchements disponibles : ['Bryozoa' 'Arthropoda' 'Cnidaria' 'Echinodermata' 'Mollusca' 'Chordata'
 'Tracheophyta' 'Rhodophyta' 'Bryophyta' 'Annelida' 'Chlorophyta'
 'Marchantiophyta' 'Porifera' 'Platyhelminthes' 'Charophyta'
 'Xenacoelomorpha' 'Sipuncula' 'Ctenophora' 'Nemertea' 'Phoronida'
 'Rotifera' 'Tardigrada' 'Nematoda' 'Anthocerotophyta' 'Orthonectida'
 'Brachiopoda' 'Chaetognatha' 'Acanthocephala' 'Nematomorpha' nan
 'Kinorhyncha' 'Gastrotricha' 'Glaucophyta' 'Hemichordata' 'Priapulida']


                          Moyenne  Médiane  Minimum  Maximum  1er décile  \
nombreObs par maille        11290     8886     2409   258788        5620   
nombreEspeces par maille     1513

In [16]:
# Chargement de fichiers SIG spécifiques à la France

# Chemins des fichiers SIG
bioregion_fichier = os.path.join(sig_path,"SIG_local","France", "region_biogeographique.shp")
departement_fichier = os.path.join(sig_path,"SIG_local","France", "carte_departements.geojson")
PNR_fichier = os.path.join(sig_path,"SIG_local","France", "N_ENP_PNR_S_000.shx")
PN_fichier = os.path.join(sig_path,"SIG_local","France", "N_ENP_PN_S_000.shx")
RNR_Occitanie_fichier = os.path.join(sig_path,"SIG_local","France", "reserves-naturelles-regionales-rnr.geojson")
RNR_PACA_fichier = os.path.join(sig_path,"SIG_local","France", "reserves-naturelles-regionales-rnr.geojson")
APPB_fichier = os.path.join(sig_path,"SIG_local","France", "N_ENP_APB_S_R93.shp")

# Chargement des fichiers SIG
bioregion_gpd = gpd.read_file(bioregion_fichier)  # Régions biogéographiques
departement_gpd = gpd.read_file(departement_fichier)  # Départements
RNR_Occitanie_gpd = gpd.read_file(RNR_fichier)  # RNR
RNR_PACA_gpd = gpd.read_file(RNR_fichier)  # RNR
APPB_gpd = gpd.read_file(APPB_fichier)  # APPB
PNR_gpd = gpd.read_file(PNR_fichier)[['NOM_SITE', 'geometry']]  # Parcs Naturels Régionaux
PN_gpd = gpd.read_file(PN_fichier)[['NOM_SITE', 'geometry']]  # Parcs Nationaux
print("✅ Chargement des fichiers SIG terminés.")

# Suppression des mentions "aire d'adhésion" dans les noms des Parcs Nationaux
PN_gpd["NOM_SITE"] = PN_gpd["NOM_SITE"].str.replace(r"\s*\[aire d'adhésion\]\s*", "", regex=True)
PN_gpd["NOM_SITE"] = PN_gpd["NOM_SITE"].str.replace(r"\s*\[Aire d'adhésion\]\s*", "", regex=True)

# Fusionner les géométries par nom de site (dissolve)
PN_gpd_fusionne = PN_gpd.dissolve(by="NOM_SITE")

# Réinitialisation de l'index après fusion
PN_gpd_fusionne.reset_index(inplace=True)

# Fusionner les Parcs Nationaux et Régionaux dans un même GeoDataFrame
PN_et_PNR_gpd = pd.concat([PNR_gpd, PN_gpd_fusionne], axis=0)
PN_et_PNR_gpd = PN_et_PNR_gpd.sort_values(by='NOM_SITE', ascending=True)
PN_et_PNR_gpd = PN_et_PNR_gpd.reset_index(drop=True)



✅ Chargement des fichiers SIG terminés.


In [17]:
#Filtrer les données selon un espace géographique
import warnings
warnings.filterwarnings("ignore", message="Geometry is in a geographic CRS")

print(f"Avant")
print(f"Nombre de mailles dans la zone selectionnées : {carte_maille_raw[cle_geo].nunique()}")
print(f"\nNombre de mailles avec données : {df_raw_import[cle_geo].nunique()}")
print(f"\nNombre totale d'observation : {df_raw_import['nombreObs'].sum()}")

filtrage_geo = True
filtrage_site=True
zone_name_short = "Sud Med"
type_site=departement_gpd
site = regions_france[zone_name_short]
cle_nom_site='code'
taux_min=0.2

liste_mailles = []

if filtrage_site:  # Vérifie que la liste n'est pas vide
    for s in site:
        mailles = lister_mailles_dans_site(
            carte_maille_raw, type_site, s,
            taux_min=taux_min, cle_geo=cle_geo, cle_nom_site=cle_nom_site, methode='exact'
        )
        liste_mailles.extend(mailles)

    # Élimine les doublons si nécessaire
    liste_mailles = list(set(liste_mailles))
        
if filtrage_geo:
    df_biodiv_periode = df_raw_import[df_raw_import[cle_geo].isin(liste_mailles)]
    carte_maille=carte_maille_raw[carte_maille_raw[cle_geo].isin(liste_mailles)]


zones_locales = site

if zones_locales:
    Zone_gpd_filtered = type_site[type_site[cle_nom_site].isin(zones_locales)]

print(f"Après")
print(f"Nombre de mailles dans la zone selectionnées : {carte_maille[cle_geo].nunique()}")
print(f"\nNombre de mailles avec données : {df_biodiv_periode[cle_geo].nunique()}")
print(f"\nNombre totale d'observation : {df_biodiv_periode['nombreObs'].sum()}")

# Création du DataFrame sans période (somme globale)
df_biodiv_sansperiode = df_biodiv_periode.groupby([cle_geo, cle_ID], as_index=False)["nombreObs"].sum()
df_biodiv_sansperiode = df_biodiv_sansperiode.merge(dico_taxo, on=cle_ID, how="left")

Avant
Nombre de mailles dans la zone selectionnées : 15523

Nombre de mailles avec données : 15523

Nombre totale d'observation : 176625890
Après
Nombre de mailles dans la zone selectionnées : 3208

Nombre de mailles avec données : 3208

Nombre totale d'observation : 40481930


In [ ]:
resultats = []

# Boucle sur chaque département
for nom_departement in departement_gpd['nom']:
    liste_mailles = filtrer_geographie_zone(
        df_raw_import, 
        carte_maille,
        type_site=departement_gpd, 
        site=[nom_departement], 
        cle_geo=cle_geo, 
        cle_nom_site="nom", 
        taux_min=0.5
    )
    
    resultats.append({
        "nom": nom_departement,
        "liste_mailles": liste_mailles
    })

# Transformation en DataFrame
df_mailles_par_departement = pd.DataFrame(resultats)

resultats = []

# Boucle sur chaque département
for nom_PN in PN_et_PNR_gpd['NOM_SITE']:
    liste_mailles = filtrer_geographie_zone(
        df_raw_import, 
        carte_maille,
        type_site=PN_et_PNR_gpd, 
        site=[nom_PN], 
        cle_geo=cle_geo, 
        cle_nom_site="NOM_SITE", 
        taux_min=0.5
    )
    
    resultats.append({
        "nom": nom_PN,
        "liste_mailles": liste_mailles
    })

# Transformation en DataFrame
df_mailles_par_PN = pd.DataFrame(resultats)


In [ ]:

df_sud = df_biodiv[df_biodiv[cle_geo].isin(liste_mailles)]

###  Normalisation des données

In [18]:

def normaliser_all_in_one(df_biodiv,carte_maille, cle_ID, cle_geo,size_grid, col='nombreObs'):
    """
    Fonction pour filtrer et normaliser les données de biodiversité.
    
    Paramètres :
    - df_biodiv : DataFrame des observations
    - cle_ID : Clé d'identification des espèces
    - cle_geo : Clé géographique des mailles
    - col : Colonne à normaliser (par défaut 'nombreObs')

    Retourne :
    - df_biodiv : DataFrame normalisé
    """

    # Normalisation initiale des données
    df_biodiv_norm = normaliser_unique(df_biodiv)  

    # Normalisation selon différentes échelles
    df_biodiv = normaliser_par_espece(df_biodiv, cle_ID, col)
    df_biodiv_norm=normaliser_par_aire_et_clade(df_biodiv_norm, carte_maille, cle_geo=cle_geo,size_grid=size_grid, clade_col='species', observation_col=col)
    df_biodiv_norm=normaliser_par_aire_et_clade(df_biodiv_norm, carte_maille, cle_geo=cle_geo,size_grid=size_grid, clade_col='phylum', observation_col=col)
    df_biodiv_norm=normaliser_par_aire_et_clade(df_biodiv_norm, carte_maille, cle_geo=cle_geo,size_grid=size_grid, clade_col='kingdom', observation_col=col)
    #df_biodiv_norm=normaliser_par_maille_et_clade(df_biodiv_norm, cle_geo=cle_geo, clade_col='kingdom', observation_col=col)
    
    # Normalisation logarithmique
    df_biodiv_norm = normaliser_log(df_biodiv_norm, 'nombreObs_norm_par_maille_et_kingdom')
    df_biodiv_norm = normaliser_log(df_biodiv_norm, 'nombreObs_norm_par_maille_et_phylum')
    df_biodiv_norm = normaliser_log(df_biodiv_norm, col)

    # Extraction des colonnes contenant 'nombreObs'**
    liste_nombres = [col for col in df_biodiv_norm.columns if 'nombreObs' in col]

    return df_biodiv_norm

# Application de la fonction
df_biodiv = normaliser_all_in_one(df_biodiv_sansperiode,carte_maille, cle_ID, cle_geo,size_grid=grid_size_km)

print("✅ Normalisations terminées")

Le nombre d observations total par espèce est fixé à 10 000
Les aires sont normalisées à 2 km2.
Il y a en moyenne 38.4 observations par species et par km2.
Les aires sont normalisées à 2 km2.
Il y a en moyenne 6153.5 observations par phylum et par km2.
Les aires sont normalisées à 2 km2.
Il y a en moyenne 14965.3 observations par kingdom et par km2.
✅ Normalisations terminées


## EXPLORATION DES DONNEES  

In [30]:

map_interactive = afficher_carte_interactive(carte_maille,cle_geo)
#map_interactive.save("ma_carte.html")  # Pour l'enregistrer
map_interactive 

# 10kmE00694N04801FRA 10kmE00707N04801FRA

In [ ]:
#Calculer la surface de la zone maillée en km2
carte_maille = carte_maille.to_crs(epsg=4326)
carte_maille['aire'] = carte_maille.geometry.area
    
max_aire = carte_maille['aire'].max()  # Trouver la valeur maximale de l'aire
carte_maille['aire'] = (carte_maille['aire'] / max_aire) * (grid_size_km ** 2)
print(f"La superficie du territoire maillé est de {round(carte_maille['aire'].sum())} km2.")


In [26]:
cols = [col for col in df_biodiv.columns if 'nombreObs' in col]
print(cols)


['nombreObs', 'nombreObs_unique', 'nombreObs_norm_par_espece', 'nombreObs_norm_par_maille_et_species', 'nombreObs_norm_par_maille_et_phylum', 'nombreObs_norm_par_maille_et_kingdom', 'nombreObs_norm_par_maille_et_kingdom_log', 'nombreObs_norm_par_maille_et_phylum_log', 'nombreObs_log']


### Afficher le top des espèces les plus observées

In [51]:
# Sélectionner les données
df_filt = df_biodiv.copy()
df_filt=df_filt[df_filt[cle_geo]=="2km_373_116_FRA"] #Pourra
#df_filt=df_filt[(df_filt[cle_geo]=="2km_375_109_FRA")|(df_filt[cle_geo]=="2km_374_112_FRA")] #Bonnieu
df_filt=df_filt[df_filt['nombreObs']>=5] 

#df_filt = filtrer_categorie(df_filt, 'arthropodes',filtres)
#df_filt = df_filt[df_filt['kingdom'] == 'Plantae']
#df_filt=filtrer_top(df_filt, 'nombreObs', 10, cle_ID)

# Afficher le nombre d'espèces uniques observées
print(f"{df_filt['nombreObs'].sum()} observations")
print(f"{df_filt[cle_ID].nunique()} espèces")

# Définir la colonne des valeurs
col_valeur = 'nombreObs_norm_par_espece'

# Afficher le top des espèces les plus observées
top_espece = afficher_top_especes(df_filt, dico_taxo, col_valeur, cle_ID)
afficher_dataframe(top_espece, [col_valeur] + liste_col_taxo, col_valeur).head(50)


11627 observations
267 espèces


,nombreObs_norm_par_espece,speciesKey,species,vernacularName_fr,vernacularName_en,genus,family,order,class,phylum,kingdom,occurrenceID
0,"1,851.85",2891720,Oxalis incarnata,NaN,Pale Pink-Sorrel,Oxalis,Oxalidaceae,Oxalidales,Magnoliopsida,Tracheophyta,Plantae,5d2978b1-5c9e-4558-abcb-5298be272b7b
1,"1,714.29",4363583,Rapana venosa,NaN,Veined Whelk,Rapana,Muricidae,Neogastropoda,Gastropoda,Mollusca,Animalia,49b3dede-9580-4a52-b1f6-8f4956f96a53
2,923.08,6126571,Haloa japonica,NaN,Japanese Bubble Snail,Haloa,Haminoeidae,Cephalaspidea,Gastropoda,Mollusca,Animalia,49501a7a-4005-42b7-971c-79144054807f
3,222.22,4569531,Papillifera solida,NaN,NaN,Papillifera,Clausiliidae,Stylommatophora,Gastropoda,Mollusca,Animalia,60A32C9A-16E2-49EC-A5C0-660D1F44090E
4,133.33,8029136,Helix melanostoma,NaN,NaN,Helix,Helicidae,Stylommatophora,Gastropoda,Mollusca,Animalia,https://observation.org/observation/264437017
5,120.35,2493128,Acrocephalus arundinaceus,Rousserolle Turdoïde,Great Reed Warbler,Acrocephalus,Acrocephalidae,Passeriformes,Aves,Chordata,Animalia,64c3849c-fe07-4be5-a1f1-0da582578e24
6,108.93,2295438,Limax maximus,NaN,Great Grey Slug,Limax,Limacidae,Stylommatophora,Gastropoda,Mollusca,Animalia,0840d1c4-7b80-11e7-8d9c-005056010096
7,89.93,5211367,Salaria pavo,Blennie Paon,Peacock Blenny,Salaria,Blenniidae,Perciformes,NaN,Chordata,Animalia,ce543e24-685a-4df3-9033-3fdea2398104
8,83.19,7627205,Glebionis coronaria,NaN,Daisy,Glebionis,Asteraceae,Asterales,Magnoliopsida,Tracheophyta,Plantae,2e9c7132-ebb6-4143-873d-8778ed15910a
9,73.64,2493118,Acrocephalus scirpaceus,Rousserolle Effarvatte,Eurasian Reed Warbler,Acrocephalus,Acrocephalidae,Passeriformes,Aves,Chordata,Animalia,fc980085-da04-4ab4-ad5a-f4d15d3903b3


In [52]:
top_espece

,speciesKey,nombreObs_norm_par_espece,species,vernacularName_fr,vernacularName_en,genus,family,order,class,phylum,kingdom,occurrenceID
0,2891720,"1,851.85",Oxalis incarnata,NaN,Pale Pink-Sorrel,Oxalis,Oxalidaceae,Oxalidales,Magnoliopsida,Tracheophyta,Plantae,5d2978b1-5c9e-4558-abcb-5298be272b7b
1,4363583,"1,714.29",Rapana venosa,NaN,Veined Whelk,Rapana,Muricidae,Neogastropoda,Gastropoda,Mollusca,Animalia,49b3dede-9580-4a52-b1f6-8f4956f96a53
2,6126571,923.08,Haloa japonica,NaN,Japanese Bubble Snail,Haloa,Haminoeidae,Cephalaspidea,Gastropoda,Mollusca,Animalia,49501a7a-4005-42b7-971c-79144054807f
3,4569531,222.22,Papillifera solida,NaN,NaN,Papillifera,Clausiliidae,Stylommatophora,Gastropoda,Mollusca,Animalia,60A32C9A-16E2-49EC-A5C0-660D1F44090E
4,8029136,133.33,Helix melanostoma,NaN,NaN,Helix,Helicidae,Stylommatophora,Gastropoda,Mollusca,Animalia,https://observation.org/observation/264437017
...,...,...,...,...,...,...,...,...,...,...,...,...
262,2477968,0.88,Dendrocopos major,Pic Épeiche,Great Spotted Woodpecker,Dendrocopos,Picidae,Piciformes,Aves,Chordata,Animalia,3c860e44-35ec-11ea-817a-005056968749
263,5231918,0.87,Cuculus canorus,Coucou Gris,Common Cuckoo,Cuculus,Cuculidae,Cuculiformes,Aves,Chordata,Animalia,bb5dd860-35eb-11ea-817a-005056968749
264,2770868,0.72,Aphyllanthes monspeliensis,Aphyllanthe De Montpellier,NaN,Aphyllanthes,Asparagaceae,Asparagales,Liliopsida,Tracheophyta,Plantae,81ab9781-8f28-4b9d-9e85-cedb42a24cb0
265,2879098,0.68,Quercus ilex,Chêne Vert,Evergreen Oak,Quercus,Fagaceae,Fagales,Magnoliopsida,Tracheophyta,Plantae,4ab1e1f4-d5e5-4102-8125-d8fbb2caf51a


### Chercher les espèces à partir d'un mot 

In [ ]:
df_filt=df_biodiv.copy()
#df_filt = filtrer_categorie(df_filt, 'plantes',filtres)
df_filt=df_filt[df_filt['kingdom']=='Animalia']
mot="Desman"

col_valeur='nombreObs'

resultat_recherche=chercher_espece(df_filt,dico_taxo,mot,col_valeur,cle_ID)

print(f"Nombre d'espèces correspondant au critère : {len(resultat_recherche)} espèces")
print(f"Nombre d'observations correspondant au critère : {resultat_recherche['nombreObs'].sum()} observations")

# Afficher les espèces contenant le mot cherché
afficher_dataframe(resultat_recherche,[col_valeur]+liste_col_taxo,col_valeur).head(50)

### Afficher les sous clades du clade choisi

In [ ]:
# Liste des clades INPN= ['all','regne', 'classe', 'ordre', 'famille', 'genre','nomScientifique','nomVernaculaire']
# Liste des clades GBIF= ['kingdom', 'class', 'order', 'family', 'genus','species']
clade = 'phylum'
taxon='Chordata'
df_filt = df_biodiv.copy()
#df_filt=filtrer_top_global(df_filt, 'nombreObs', 10, cle_ID)


explorer_clade(df_filt,clade,taxon,cle_ID,'nombreEspèces') #colorder = 'nombreEspèces' ou 'nombreObs'  ou 'Ratio Obs/Esp'

In [ ]:

nb_unique = df_filt["species"].nunique()
print(nb_unique)

In [ ]:
##A REFAIRE#### Afficher la carte avec les mailles

afficher_carte_maille(carte_maille)

In [ ]:
##A REFAIRE#### Filtrer une liste de mailles à étudier
liste_codes=['10kmL93E089N623']
col_valeur='nombreObs_norm_par_maille_et_regne'

df_filt=df_inpn[df_inpn[cle_geo].isin(liste_codes)]
df_maille=df_filt.groupby(cle_ID)[col_valeur].sum()

df_dico=generer_dictionnaire_taxonomie(df_filt,cle_ID)
df_maille=pd.merge(df_maille,df_dico,on=cle_ID)

afficher_dataframe(df_maille,[col_valeur,cle_ID,'nomVernaculaire','nomScientifique','famille','ordre','classe','regne','especeProtegee'],col_sort=col_valeur).head(10)

In [ ]:
##A REFAIRE#### Rechercher les espèces protégées présentes dans une zone
col_valeur='nombreObs_norm_par_espece'
liste_codes=liste_geo_PNR_Vosges

chercher_especes_protegees(df_inpn,liste_codes,cle_geo='codeMaille10Km',cle_ID='cdRef',col_valeur=col_valeur)

afficher_dataframe(grouped_local_especes_protegee,[col_valeur,cle_ID,'nomVernaculaire','nomScientifique','famille','ordre','classe','regne','especeProtegee']).head(10)

## AFFICHAGE DES DONNEES SOUS FORME DE CARTE 

### Préparation de l'affichage

In [36]:
# Configuration de la carte par défaut
fond_de_carte="Sud"
def charger_couches_SIG_defaut(fig,ax):  
    fig, ax = ajouter_couche_SIG(fig, ax, carte_maille,
                             facecolor='none',alpha=0.3,
                             edgecolor="white",linewidth=0.5,linestyle='--',
                             with_label=False,col_label='code',label_color="white",fontsize=8)
                                     
    fig, ax = ajouter_couche_SIG(fig, ax, departement_gpd,
                             facecolor='none',alpha=1,
                             edgecolor="white",linewidth=0.5,linestyle='-',
                             with_label=False,col_label='code',label_color="white",fontsize=8)
    

                                     
    # fig, ax = ajouter_couche_SIG(fig, ax, border_geo,
    #     facecolor='none',alpha=0.5,
    #     edgecolor="white",linewidth=0.5,linestyle='-',
    #     with_label=False,col_label='code',label_color="white",fontsize=8)

    """ Liste de couches SIG disponibles :
    PNR_gpd_filtered
    border_local_geo
    departement_gpd
    bioregion_gpd
    """
    return fig, ax
    
fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='Esri')
fig, ax = charger_couches_SIG_defaut(fig, ax)
plt.show()

In [123]:
ax = fig.gca()
x_centre = (ax.get_xlim()[0] + ax.get_xlim()[1]) / 2
y_centre = (ax.get_ylim()[0] + ax.get_ylim()[1]) / 2
y_lim=ax.get_ylim()[1] 

print("Centre des données : x={:.2e}, y={:.2e}, lim={:.2e}".format(x_centre, y_centre,y_lim))


Centre des données : x=3.66e+05, y=5.48e+06, lim=5.64e+06


In [125]:
# Définition des paramètres de la carte
center_x = x_centre  # Coordonnée X du centre de la carte
center_y = y_centre   # Coordonnée Y du centre de la carte
height = (y_lim- center_y) * 2  # Hauteur de la carte
size_x = 15          # Taille de la figure en X
size_y = 9         # Taille de la figure en Y
zoom_size = 8       # Niveau de zoom

# Configuration de la carte avec OpenStreetMap
fig, ax = configurer_carte('Esri', center_x, center_y, height, zoom=zoom_size, fig_size=(size_x, size_y))

# Ajout de la couche des départements avec des bordures blanches en pointillés
#fig, ax = ajouter_couche_SIG(fig, ax, Zone_gpd_filtered, linewidth=2, edgecolor='black', linestyle='--')

# Affichage de la carte
plt.show()


In [126]:
# Ajouter une entrée au dictionnaire
nouvelle_carte = "Cévennes"
paramètres = {
    "center_x": center_x,
    "center_y": center_y,
    "height": height,
    "size_x": size_x,
    "size_y": size_y,
    "zoom": zoom_size
}

# Ajouter la nouvelle entrée
dictionnaire_cartes[nouvelle_carte] = paramètres

# Sauvegarder le dictionnaire mis à jour
with open(chemin_dico_cartes, "w", encoding="utf-8") as fichier:
    json.dump(dictionnaire_cartes, fichier, indent=4)

print(f"L'entrée '{nouvelle_carte}' a été ajoutée et enregistrée.")

L'entrée 'Cévennes' a été ajoutée et enregistrée.


In [18]:
# Afficher la liste des instances
instances = list(dictionnaire_cartes.keys())

print("Instances disponibles dans dictionnaire_cartes :")
for instance in instances:
    print("-", instance)

Instances disponibles dans dictionnaire_cartes :
- France
- Madagascar
- PN Cévennes
- Ballons des Vosges
- Allemagne
- Amérique Sud
- Brazil
- Maghreb
- Méditerranée
- Philippines
- Afrique de l'Est
- Chine
- Iran
- Japon+Corée
- Espagne-France-Italie
- Costa Rica
- Monde
- Sri Lanka
- Amérique Centrale
- Asie du Sud-Est
- Asie du Sud+Sud-Est
- Italie
- Espagne
- Europe
- Test
- Sud
- Asie du Sud-Est_red
- Mekong - South East Asia
- Gabon
- Pyrénées


### Affichage de la carte pour un taxon

In [38]:
#Afficher la carte de répartition d'un taxon

col_valeur = 'nombreObs' #'nombreObs_norm_par_maille_et_kingdom' ,'nombreObs_unique' , 'nombreObs'
colormap = 'viridis'
#fond_de_carte="Espagne-France-Italie"
log_values = True
SIG_defaut=True
save_carte=True
predit=False


    
# Sélection du taxon et du clade
clade = 'species'  # Peut être : nomScientifique, nomVernaculaire, regne, classe, ordre, famille, genre
taxon = 'Colchicum filifolium'

# Définition du titre de la figure
titre = f'Carte de répartition de {taxon} dans {zone_name_short}'
#titre = 'Carte de répartition de' #Pour entrée manuelle

# Filtrage des données
df_filt = df_biodiv[df_biodiv[clade] == taxon]

if predit:
    df_filt = df_complet_predit[df_complet_predit[clade] == taxon]
    col_valeur = f"{col_valeur}_predit"

# Affichage du fond de carte
fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='Esri')


# Ajout des données
fig, ax = ajouter_couche_continue(
    fig, ax, df_filt, carte_maille, col_valeur, cle_geo,
    quantile_inf=0.0, quantile_sup=1,
    borne_min=None, borne_max=None,
    cmap_choice=colormap, val_alpha=0.8,
    log_values=log_values
)

if SIG_defaut:
    fig, ax = charger_couches_SIG_defaut(fig, ax)
    
# Ajout de couches SIG supplémentaires
"""
fig, ax = ajouter_couche_SIG(fig, ax, departement_gpd,
                             facecolor='none',alpha=1,
                             edgecolor="grey",linewidth=1,linestyle='--',
                             with_label=True,col_label='code',label_color="white",fontsize=8
                            )
"""

# Ajout du titre et de la légende
ax.set_title(titre, fontsize=16)
fig.text(0.45, 0.15, f'var : {col_valeur}', ha='center', va='center', fontsize=10)

# Affichage de la carte
plt.show()

if save_carte:
    fig.savefig(save_path+'/'+titre+'.png', dpi=300, bbox_inches='tight')  # Enregistre au format PNG avec une résolution de 300 DPI

## INDICE DE BIODIVERSITE et ENDEMISME  

### CALCUL DES INDICES

In [122]:
# Configuration des paramètres
predic = True

if predic is True:
    col_valeur = 'nombreObs_norm_par_maille_et_kingdom_predit'
    df_filt = df_complet_predit.copy()  # Filtrer les valeurs NaN sur cle_geo
    df_filt = df_filt[df_filt[col_valeur] >= 0]

else :
    filt='global'
    n_filt=1
    col_valeur = 'nombreObs'
    # Préparation des données
    df_filt = df_biodiv.copy()  # Filtrer les valeurs NaN sur cle_geo
    #df_filt = df_biodiv[df_biodiv['order'] == 'Odonata']
    
    if filt=="global":
        df_filt = filtrer_top_global(df_filt, col_valeur, n_filt, cle_ID)  # Filtrer les 10 valeurs dont col > n
    if filt=="mailles":
        # Filtrer les n_top espèces les plus abondantes par maille
        df_filt = filtrer_top_mailles(df_filt, col_valeur, n_filt, cle_ID,cle_geo) 

# Calcul des indices de biodiversité et d'endémisme 
df_indices = calculer_indices(df_filt, col_valeur, cle_geo, cle_ID)


C:\Users\Aubin\Documents\MANTIS\Code\biodiversite_endemisme_biodiv.py:110: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_top = df_input.groupby(cle_geo).apply(lambda x: x.nlargest(n_top, col_valeur)).reset_index(drop=True)


In [ ]:
#Calculer de la diversité taxonomique
n_filt=2
col_valeur = 'nombreObs' #'nombreObs','nombreObs_norm_par_maille_et_kingdom' 'nombreObs_norm_par_maille_et_kingdom_predit'

if predic is True:
    col_valeur = 'nombreObs_predit'
    df_filt = df_complet_predit.copy()  # Filtrer les valeurs NaN sur cle_geo
    df_filt = df_filt[df_filt[col_valeur] >= 0]
    if n_filt:
        df_filt = filtrer_top_mailles(df_filt, col_valeur, n_filt, cle_ID,cle_geo) 

else :
    if n_filt:
        df_filt = filtrer_top_mailles(df_biodiv, col_valeur, n_filt, cle_ID,cle_geo) 
    
start_time = time.time()  # Temps de départ
grouped_Rao_taxo = calculer_entropie_quadratique_taxo(df_filt, col_valeur=col_valeur, cle_geo=cle_geo, cle_ID=cle_ID)
end_time = time.time()  # Temps de fin
elapsed_time = end_time - start_time

print(f"Temps d'exécution de calculer_indices : {elapsed_time:.2f} secondes")
# Merge en remplaçant la colonne si elle existe déjà
if 'entropie_quadratique_taxo' in df_indices.columns:
    df_indices['entropie_quadratique_taxo'] = grouped_Rao_taxo['entropie_quadratique_taxo']
else:
    df_indices = pd.merge(df_indices, grouped_Rao_taxo, on=cle_geo, how='left')


### MATRICE DE CORRELATION

In [84]:
#Calculer la corrélation entre les indices

cols_all = ["nombre_observations", "nombre_especes", 
        "indice_de_Shannon", 
        #"entropie_quadratique_taxo",
        "indice_de_Simpson",
        "indice_de_Margalef",
        "indice_equitabilite_simpson",
        "indice_equitabilite_heip",
        "indice_de_BergerParker",
        "indice_d_endemisme"]

cols="all"
if cols=="all":
    cols=cols_all
cols=[ "nombre_observations","nombre_especes","indice_de_Shannon","indice_d_endemisme"]

method_corr='kendall'
# Calcul de la corrélation (Kendall)
corr_kendall = df_indices[cols].corr(method=method_corr)

plt.figure(figsize=(9, 7))
ax = sns.heatmap(
    corr_kendall,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    cbar_kws={"shrink": 0.8}
)

# Rotation et mise en page des labels
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
titre = f"Corrélation ({method_corr}) entre indices_pearson_predit"
plt.title(titre, fontsize=20, pad=20)

# Ajustement auto des marges
plt.tight_layout()
plt.show()


### Comparaison obs et predit

In [46]:
#Calculer les indices pour # Configuration des paramètres
col_valeur = 'nombreObs_norm_par_maille_et_kingdom_predit'
df_filt = df_complet_predit.copy()  # Filtrer les valeurs NaN sur cle_geo
df_filt = df_filt[df_filt[col_valeur] > 0]

# Calcul des indices de biodiversité et d'endémisme 
df_indices_pred = calculer_indices(df_filt, col_valeur, cle_geo, cle_ID)

C:\Users\Aubin\Documents\MANTIS\Code\biodiversite_endemisme_biodiv.py:110: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_top = df_input.groupby(cle_geo).apply(lambda x: x.nlargest(n_top, col_valeur)).reset_index(drop=True)


In [47]:
filt=None
n_filt=50
col_valeur = 'nombreObs'
# Préparation des données
df_filt = df_biodiv.copy()  # Filtrer les valeurs NaN sur cle_geo
#df_filt = df_biodiv[df_biodiv['order'] == 'Odonata']

if filt=="global":
    df_filt = filtrer_top_global(df_filt, col_valeur, n_filt, cle_ID)  # Filtrer les 10 valeurs dont col > n
if filt=="mailles":
    # Filtrer les n_top espèces les plus abondantes par maille
    df_filt = filtrer_top_mailles(df_filt, col_valeur, n_filt, cle_ID,cle_geo) 

# Calcul des indices de biodiversité et d'endémisme 
df_indices_obs = calculer_indices(df_filt, col_valeur, cle_geo, cle_ID)

C:\Users\Aubin\Documents\MANTIS\Code\biodiversite_endemisme_biodiv.py:110: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_top = df_input.groupby(cle_geo).apply(lambda x: x.nlargest(n_top, col_valeur)).reset_index(drop=True)


In [48]:
cols_all = ["nombre_observations", "nombre_especes", 
            "indice_de_Shannon", "indice_de_Simpson",
            "indice_de_Margalef", "indice_equitabilite_simpson",
            "indice_equitabilite_heip", "indice_de_BergerParker",
            "indice_d_endemisme"]

# Comparaison avec normalisation et soustraction
df_compare_sub = comparer_indices(df_indices_pred, df_indices_obs, cle_geo, cols_all, norm=False, method="sub")

# Comparaison avec normalisation et soustraction
df_compare_sub_norm= comparer_indices(df_indices_pred, df_indices_obs, cle_geo, cols_all, norm=True, method="sub")

# Comparaison sans normalisation avec un rapport
df_compare_div = comparer_indices(df_indices_pred, df_indices_obs, cle_geo, cols_all, norm=True, method="div")

C:\Users\Aubin\Documents\MANTIS\Code\biodiversite_endemisme_biodiv.py:366: RuntimeWarning: divide by zero encountered in divide
  df_compare[f'{col}'] = np.where(col_obs != 0, col_pred / col_obs, np.nan)
C:\Users\Aubin\Documents\MANTIS\Code\biodiversite_endemisme_biodiv.py:366: RuntimeWarning: divide by zero encountered in divide
  df_compare[f'{col}'] = np.where(col_obs != 0, col_pred / col_obs, np.nan)
C:\Users\Aubin\Documents\MANTIS\Code\biodiversite_endemisme_biodiv.py:366: RuntimeWarning: divide by zero encountered in divide
  df_compare[f'{col}'] = np.where(col_obs != 0, col_pred / col_obs, np.nan)
C:\Users\Aubin\Documents\MANTIS\Code\biodiversite_endemisme_biodiv.py:366: RuntimeWarning: divide by zero encountered in divide
  df_compare[f'{col}'] = np.where(col_obs != 0, col_pred / col_obs, np.nan)
C:\Users\Aubin\Documents\MANTIS\Code\biodiversite_endemisme_biodiv.py:366: RuntimeWarning: divide by zero encountered in divide
  df_compare[f'{col}'] = np.where(col_obs != 0, col_pred

### AFFICHAGE DE LA CARTE

In [128]:
# Paramètrage de la carte à afficher
predic=True
compare=False
#zone_name_short = "Monde"
colormap = 'plasma'
fond_carte = 'Esri'
if compare is True:
    df_indices_choisi = df_compare_div
else : df_indices_choisi = df_indices

log_values = False
SIG_defaut=True
save=True

cols_all = ["nombre_observations", "nombre_especes", 
        "indice_de_Shannon", 
        #"entropie_quadratique_taxo",
        "indice_de_Simpson",
        "indice_de_Margalef",
        "indice_equitabilite_simpson",
        "indice_equitabilite_heip",
        "indice_de_BergerParker",
        "indice_d_endemisme"]

cols="all"
cols=["nombre_observations", "nombre_especes","indice_de_Shannon","indice_d_endemisme"]

if cols=="all":
    cols=cols_all

for indice_choisi in cols:
    # Définition du titre de la figure
    titre = f"{indice_choisi}_{zone_name_short}"
    if predic is True:
        titre = f"{indice_choisi}_{zone_name_short}_predit"
    if compare is True:
        titre = f"{indice_choisi}_{zone_name_short}_comparaison"
    
    #titre = 'Carte de ' #Pour entrée manuelle
    
    # Affichage du fond de carte
    fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond=fond_carte)
    
    # Ajout des couches
    fig, ax = ajouter_couche_continue(
        fig, ax, df_indices_choisi, carte_maille, indice_choisi, cle_geo,
        quantile_inf=0.01, quantile_sup=0.99, borne_min=None, borne_max=None,
        cmap_choice='plasma', val_alpha=0.7, missing_color='black', 
        colorbar_choice=True, log_values=log_values,sig=5
    )
    
    if SIG_defaut:
        fig, ax = charger_couches_SIG_defaut(fig, ax)
    
    # Ajout du titre
    ax.set_title(titre, fontsize=16)
    
    if predic is True:
        fig.text(0.45, 0.15, f'var : {col_valeur}, nmin_global : {nmin_filt_predic}, ntop_mailles: {ntop_filt_predic}', ha='center', va='center', fontsize=10)
        if save:
            fig.savefig(f"{save_path}/{titre}_{nmin_filt_predic}_{ntop_filt_predic}.png", dpi=300, bbox_inches='tight')
    else : 
        fig.text(0.45, 0.15, f'var : {col_valeur}, n_filt : {n_filt}', ha='center', va='center', fontsize=10)
        if save:
            fig.savefig(f"{save_path}/{titre}_{n_filt}.png", dpi=300, bbox_inches='tight')
    # Affichage de la carte
    plt.show()


### Calcul des stats par zone

In [ ]:
cols = [
    "nombre_observations", "nombre_especes", 
    "indice_de_Shannon","indice_d_endemisme"
]

critere = "q5"  # "moy", "q5" ou "q9"
df_indices_par_zone, liste_colonnes = calcul_indices_par_zone(
    df_zone=df_mailles_par_PN, #df_mailles_par_PN df_mailles_par_departement
    df_indices=df_indices, 
    cle_geo=cle_geo,  # à adapter selon ton DataFrame
    cols=cols,
    critere=critere
)

# Affichage avec ta fonction
afficher_dataframe(
    df_indices_par_zone,
    liste_colonnes=liste_colonnes,
    col_sort='rang_moyen',
    ascending=True,
    n_rows=None,
    n_dec=2
)

## DETERMINATION DE BIOREGIONS - ANALYSE PAR CLUSTERS GEOGRAPHIQUES

### Préparation

In [162]:
# Paramètres 
predic=True
col_valeur='nombreObs_norm_par_maille_et_kingdom'
log_values = True 

filtre="plantes"
filtre=None

# ne s'applique pas si predic = True :
method="global"
var_global='nombreObs'
nmin_filt = 100
n_filt=100
var_mailles = 'nombreObs'
ntop_filt=10


In [163]:
# Analyse en composantes principales

#Paramètres de la PCA
variance_threshold=0.99
n_components=None
max_components=1000

if predic:
    df_filt=df_complet_predit.copy()
else : 
    df_filt=df_biodiv.copy()
    
if filtre is not None:
    df_filt = filtrer_categorie(df_filt, filtre,filtres)
    
# Filtrage des données
if predic:
    statut_predic = "avec"
    col_valeur=col_valeur+"_predit"
else:
    statut_predic = "sans"
    #df_filt = df_filt[df_filt['order'] == 'Odonata']
    df_filt= filtrer_top(df_filt,method, 
                              var_global=var_global,var_mailles=var_mailles,
                              ntop_mailles=ntop_filt,nmin_global=nmin_filt, 
                              cle_ID=cle_ID,cle_geo=cle_geo)
    
print(f"Nombre d'espèces prises en compte : {len(df_filt['speciesKey'].unique())}")

if log_values:
    col_valeur=col_valeur+"_log"

# Réduction de dimension avec PCA
df_pca = analyser_composantes_principales(
    df_filt, cle_geo, col_valeur, cle_ID,
    variance_threshold=variance_threshold, n_components=n_components, max_components=max_components
)

Nombre d'espèces prises en compte : 3198
Nombre de composantes nécessaires pour atteindre 99.0% de variance expliquée: 352
Variance expliquée par les dix premières composante: [63.5 14.2  3.9  2.8  1.9  1.2  1.   0.7  0.6  0.5]
Variance cumulée totale : 99.0 %


In [141]:
# faire varier le nombre de cluster pour en déterminer le nombre optimal
determiner_k(df_pca,n_init=5,max_cluster=20) 

In [164]:
# Clustering 
# Paramètres du clustering
k =7  # Nombre de clusters
n_init = 30# Nombre de répétitions pour choisir la meilleure solution
methode_clustering = 'kmeans'  # Méthode : 'kmeans' ou 'ward'
display=False

#Paramétrage du critère de contiguité spatiale
λ = 0# Pénalité spatiale 
methode_contiguite = "euclidean"# Métrique : 'neighbor' ou 'squared euclidean' "euclidean"
if λ == 0:
    methode_contiguite = "None"
parameter = 100  # Paramètre pour la méthode 'neighbor'
n_components = df_pca.shape[1]

# Déterminer les clusters
if λ == 0:
    df_cluster = former_cluster_biogeo(df_pca,cle_geo, method=methode_clustering, k_cluster=k, n_init=n_init,display=display)
elif λ > 0:
    df_cluster = former_cluster_biogeo_avec_critere_spatial(
        df_pca, carte_maille, cle_geo, n_components, k, methode_contiguite, λ, n_init, parameter
    )
else:
    print('Erreur : Lambda doit être >= 0')


### Affichage de la carte des biorégions

In [180]:
# Configuration de la carte
col_valeur_cluster = 'Cluster'
colormap = 'Spectral_r'  # Autres options : 'Spectral_r (n)', 'Accent' (8), Pastel (9), Set2 (8), Set1 (9), Set3 (12) Paired (12) tab20b
#fond_de_carte="Italie"
SIG_defaut = True
save_carte = True

# Définition du titre de la figure
countries_name = "_".join(country.replace(" ", "_") for country in countries)

if filtre is not None:
    titre = f"Carte des biorégions de {zone_name_short} - {filtre} {statut_predic} prédiction"
else: titre = f"Carte des biorégions de {zone_name_short} - {statut_predic} prédiction"

# Affichage du fond de carte
fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='Esri') #GeoportailSatellite OpenStreetMap

# Ajout des données
fig, ax = ajouter_couche_discrete(
    fig, ax, df_cluster, carte_maille, col_valeur=col_valeur_cluster, cle_geo=cle_geo,
    cmap_choice=colormap, val_alpha=0.7,
    missing_color=None, legend_choice=True, loc_legend=3, pos_legend=None
)

if SIG_defaut:
    fig, ax = charger_couches_SIG_defaut(fig, ax)
    

# Ajout du titre et de la légende
ax.set_title(titre, fontsize=18)
fig.text(0.5, 0.09, f'variable : {col_valeur}, {cle_geo}, n_pca : {n_components}, méthode de clustering {methode_clustering},k : {k}, λ: {λ}, n_init : {n_init}, méthode de contiguité : {methode_contiguite}',
         ha='center', va='center', fontsize=10)

# Sauvegarde et affichage de la carte
if save_carte:
    # Génération du nom de fichier avec tous les paramètres
    nom_fichier = f"Carte_{zone_name_short}"
    if filtre is not None:
        nom_fichier += f"_{filtre}"
    nom_fichier += f"_{statut_predic}"
    if log_values:
        nom_fichier += "_log"
    nom_fichier += f"_pca{n_components}_predic_k{k}_λ{λ}_{methode_clustering}_{methode_contiguite}.png"
    nom_fichier = nom_fichier.replace(" ", "_")
    fig.savefig(save_path + '/' + nom_fichier + '.png', dpi=300, bbox_inches='tight')

plt.show()

### Cartes des PCA

In [150]:
# Configuration de la carte des PCA

max_pca=min(5,df_pca.shape[1])
colormap = 'plasma'  # Autres options : 'Spectral_r', 'plasma', 'Paired', 'Set1_r', 'Set2_r', 'Set3', 'tab10', 'Accent' (8)
#fond_de_carte="Asie du Sud-Est_red"
SIG_defaut = None
save_carte = True

# Définition du titre de la figure
countries_name = "_".join(country.replace(" ", "_") for country in countries)

for i in range(1,max_pca+1):
    col_valeur_PCA = 'PC'+str(i)
    if filtre is not None:
        titre = f"Carte de la PCA {i} de {zone_name_short} - {filtre} {statut_predic} prédiction"
    else: titre = f"Carte de la PCA {i} de {zone_name_short} - {statut_predic} prédiction"
    
    # Affichage du fond de carte
    fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='OpenStreetMap',zoom_manuel=5) #GeoportailSatellite OpenStreetMap
    
    # Ajout des données
    fig, ax = ajouter_couche_continue(
        fig, ax, df_pca, carte_maille, col_valeur_PCA, cle_geo,
        quantile_inf=0, quantile_sup=1,
        borne_min=None, borne_max=None,
        cmap_choice=colormap, val_alpha=0.8,
        log_values=False
    )
    
    if SIG_defaut:
        fig, ax = charger_couches_SIG_defaut(fig, ax)
    
    
    
    # Ajout du titre et de la légende
    ax.set_title(titre, fontsize=18)
    fig.text(0.5, 0.09, f'variable : {col_valeur}, n_filt : {n_filt} {cle_geo}, pca_{i}',
             ha='center', va='center', fontsize=10)

        # Sauvegarde et affichage de la carte
    if save_carte:
        # Génération du nom de fichier avec tous les paramètres
        nom_fichier = f"Carte_{zone_name_short}"
        if filtre is not None:
            nom_fichier += f"_{filtre}"
        nom_fichier += f"_{statut_predic}"
        if log_values:
            nom_fichier += "_log"
        nom_fichier += f"_pca{i}_n_filt{n_filt}.png"
        nom_fichier = nom_fichier.replace(" ", "_")
        fig.savefig(save_path + '/' + nom_fichier + '.png', dpi=300, bbox_inches='tight')
    
        plt.show()

In [ ]:
# Pour inverser des composantes :
df_pca['PC2']=-df_pca['PC2']

### Analyse par cluster

In [ ]:
#Statistiques sur les clusters 

# Fusion des données avec les clusters
df_cluster_reset = df_cluster.reset_index()

#df_filt_analyse=df_biodiv.copy() #Si l'on veut reset les filtres
df_filt_analyse=df_filt.copy()

df_merged = pd.merge(df_biodiv, df_cluster_reset[[cle_geo, 'Cluster']], how='left')

# Agrégation des données par maille et cluster
df_grouped = (
    df_merged
    .groupby([cle_geo, 'Cluster'], as_index=False)
    .agg({'nombreObs': 'sum', 'nombreObs_unique': 'sum'})
)

# Calcul des statistiques par cluster
cluster_means = df_grouped.groupby('Cluster')[['nombreObs', 'nombreObs_unique']].mean()
cluster_median = df_grouped.groupby('Cluster')[['nombreObs', 'nombreObs_unique']].median()
cluster_quantile1 = df_grouped.groupby('Cluster')[['nombreObs', 'nombreObs_unique']].quantile(0.1)
cluster_quantile9 = df_grouped.groupby('Cluster')[['nombreObs', 'nombreObs_unique']].quantile(0.9)

# Création du tableau de synthèse en concaténant les résultats
stats_summary = pd.concat([cluster_means, cluster_median, cluster_quantile1, cluster_quantile9], axis=1)

# Renommage des colonnes
stats_summary.columns = [
    "Moyenne_nombreObs",
    "Moyenne_nEspece",
    "Médiane_nombreObs",
    "Médiane_nEspece",
    "1er_decile_nombreObs",
    "1er_decile_nEspece",
    "9e_decile_nombreObs",
    "9e_decile_nEspece"
]

# Conversion en entier pour un affichage propre
stats_summary = stats_summary.astype(int)

# Affichage du tableau de synthèse
print("\n", stats_summary)


In [ ]:
# Définition des paramètres
col_valeur = 'nombreObs_norm_par_espece'
seuil_filtrage = 1000  # Seuil pour filtrer les espèces surreprésentées
cluster_choice = 2  # Cluster à afficher

# Filtrage des données
df_filt_analyse = filtrer_top_global(df_filt_analyse, 'nombreObs', seuil_filtrage, cle_ID)

# Étude de la composition des clusters
cluster_composition = etudier_composition_cluster(df_filt_analyse, df_cluster, dico_taxo, col_valeur, cle_ID, cle_geo)

# Affichage des 50 premières espèces du cluster choisi
df_cluster_choice = cluster_composition[cluster_composition['Cluster'] == cluster_choice]
colonnes_affichage = ['Cluster', col_valeur] + liste_col_taxo

afficher_dataframe(df_cluster_choice, colonnes_affichage, col_valeur).head(10)
afficher_dataframe(df_cluster_choice, ['species','vernacularName_fr','nombreObs_norm_par_espece'], col_valeur).head(10)

## ANALYSE PAR CLUSTERS D'ESPECES

### Préparation

In [ ]:
# Paramètres 
col_val='nombreObs'
log_values = True

predic=False

method="global"
var_global='nombreObs'
nmin_filt = 5000

var_mailles = 'nombreObs'
ntop_filt=1

filtre='plantes'
#filtre=None

if predic:
    df_filt=df_complet_predit.copy()
else : 
    df_filt=df_biodiv.copy()
    
if filtre is not None:
    df_filt = filtrer_categorie(df_filt, filtre,filtres)
    
# Filtrage des données
if predic:
    statut_predic = "avec"
    col_val=col_val+"_predit"
else:
    statut_predic = "sans"
    #df_filt = df_filt[df_filt['order'] == 'Odonata']
    df_filt= filtrer_top(df_filt,method, 
                              var_global=var_global,var_mailles=var_mailles,
                              ntop_mailles=ntop_filt,nmin_global=nmin_filt, 
                              cle_ID=cle_ID,cle_geo=cle_geo)
    
print(f"Nombre d'espèces prises en compte : {len(df_filt['speciesKey'].unique())}")

if log_values:
    col_val=col_val+"_log"


In [ ]:
# ---------------------------------------------------------------
# 1. Filtrage des espèces pour réduire le temps de calcul
# ---------------------------------------------------------------

import_mat_corr = True
calculate_mat_corr = False
save_mat_corr = False

df_global=df_filt.copy()

#df_global = filtrer_top(df_biodiv, 'nombreObs_norm_par_maille_et_kingdom', n_filt, cle_ID)
colonnes_obs = [col for col in df_global.columns if 'nombreObs' in col]

methode_corr = 'kendall'  # pearson, kendall


path_data_mat_corr = os.path.join(data_path, "mat_corr")
if predic is True:
    filename_mat_corr=f'mat_corr_{zone_name_short}_{"".join(regnes_a_garder)}_{nmin_filt_predic}-{ntop_filt_predic}_{col_val}_{methode_corr}_{cle_geo}.csv'
else:
    filename_mat_corr=f'mat_corr_{zone_name_short}_{"".join(regnes_a_garder)}_{nmin_filt}-{ntop_filt}_{col_val}_{methode_corr}_{cle_geo}.csv'

if import_mat_corr:
    # Chargement des données
    mat_corr = pd.read_csv(os.path.join(path_data_mat_corr, filename_mat_corr), dtype=str)
    mat_corr = mat_corr.apply(pd.to_numeric, errors='coerce')
    
elif calculate_mat_corr:
    # Jalon 1 : début du processus
    start_time = time.time()
    mat_corr = calculer_matrice_correlation(df_global, col_val, methode_corr, cle_geo, cle_ID)
    mat_corr = mat_corr.astype(float)
    end_time = time.time()
    print(f"Temps écoulé: {end_time - start_time:.2f} secondes")

# ---------------------------------------------------------------
# 2. Calcul de la matrice de corrélation et génération du dendrogramme
# ---------------------------------------------------------------
if save_mat_corr:
    # Sauvegarde de la matrice de corrélation
    mat_corr_tosave = pd.DataFrame(mat_corr)
    mat_corr_tosave.to_csv(os.path.join(path_data_mat_corr, filename_mat_corr), index=False)


In [ ]:
# ---------------------------------------------------------------
# 3.Génération du dendrogramme
# ---------------------------------------------------------------
methode_dendogram='ward' #  ward, complete
Z = generer_dendogram(mat_corr, methode='ward', display=0)  

In [ ]:
# ---------------------------------------------------------------
# 4. Formation des clusters en fonction du niveau choisi
# ---------------------------------------------------------------
#col_val = 'nombreObs_norm_par_maille_et_kingdom'
lvl = 20  # Niveau de découpage du dendrogramme
criterion = 'distance'

df_especes_cluster = former_cluster_espece(df_global, Z, col_valeur=col_val, level=lvl, crit=criterion, cle_ID=cle_ID, cle_geo=cle_geo)
df_global_cluster = pd.merge(df_global, df_especes_cluster[[cle_ID, 'Cluster_corr']], on=cle_ID)
df_global_cluster['Cluster_corr'] = pd.to_numeric(df_global_cluster['Cluster_corr'], errors='coerce').astype('Int64')

print(f'Nombre de clusters formés : {len(df_especes_cluster["Cluster_corr"].unique())}')


### Autres méthodes 

In [ ]:
import importlib
importlib.reload(clustering_espece_biodiv)


df_communautes, temps = former_communautes_all(
    df_filt,
    cle_geo=cle_geo,
    cle_ID=cle_ID,
    col_val=col_val,
    n_clusters=50,
    seuil_corr=0.2
)




In [ ]:
# df_communautes : colonnes = cle_ID + colonnes de cluster pour chaque méthode
cle_ID = 'speciesKey'  # ou ton identifiant d'espèce

# Pour chaque colonne de communauté
for col in df_communautes.columns:
    if col == cle_ID:
        continue
    # Nombre de valeurs uniques par espèce
    max_unique = df_communautes.groupby(cle_ID)[col].nunique().max()
    print(f"Colonne {col}: max communautés observées pour une espèce = {max_unique}")


In [ ]:
import pandas as pd
import numpy as np
import time

def clustering_fuzzy_multi(df_fil, n_clusters=5, m=2, seuil=0.3,
                           cle_geo='codeMaille10Km', cle_ID='cdRef', col_val='nombreObs'):
    """
    Clustering flou (Fuzzy C-Means) permettant qu'une espèce appartienne à plusieurs communautés.
    Retourne un DataFrame binaire espèces x clusters (1 = appartenance).
    """
    import skfuzzy as fuzz

    # Pivot : lignes = mailles, colonnes = espèces
    pivot_df = creer_pivot(df_fil, cle_geo, cle_ID, col_val)
    X = pivot_df.T.values

    # Supprimer les espèces vides
    non_empty = X.sum(axis=1) > 0
    X = X[non_empty, :]
    cols_kept = pivot_df.columns[non_empty]

    if X.shape[0] == 0:
        raise ValueError("Aucune espèce observée après filtrage des colonnes vides.")

    # Fuzzy C-Means
    u, _, _, _, _, _, _ = fuzz.cluster.cmeans(
        X, c=n_clusters, m=m, error=0.005, maxiter=1000, init=None
    )

    # Appliquer le seuil pour assigner à plusieurs communautés
    membership = (u > seuil).astype(int)  # True = appartenance
    cluster_cols = [f"Cluster_Fuzzy_{i+1}" for i in range(n_clusters)]

    df_clusters = pd.DataFrame(membership.T, columns=cluster_cols)
    df_clusters[cle_ID] = cols_kept.astype(str)

    return df_clusters


def former_communautes_all_multi(df_fil, cle_geo='codeMaille10Km', cle_ID='cdRef', col_val='nombreObs',
                                 n_clusters=5, seuil_corr=0.3, m_fuzzy=2, seuil_fuzzy=0.3):
    """
    Forme les communautés avec plusieurs méthodes et permet qu'une espèce appartienne à plusieurs communautés.
    Affiche le temps de calcul de chaque méthode.
    """
    df_fil = df_fil.copy()
    df_fil[cle_ID] = df_fil[cle_ID].astype(str)
    df_final = pd.DataFrame({cle_ID: df_fil[cle_ID].unique()})
    temps = {}

    # --- Fuzzy multi
    try:
        print("▶ Fuzzy multi...")
        t0 = time.time()
        df_fuzzy = clustering_fuzzy_multi(df_fil, n_clusters=n_clusters, m=m_fuzzy, seuil=seuil_fuzzy,
                                          cle_geo=cle_geo, cle_ID=cle_ID, col_val=col_val)
        df_final = df_final.merge(df_fuzzy, on=cle_ID, how='left')
        temps['Fuzzy'] = round(time.time() - t0, 3)
    except ModuleNotFoundError:
        print("⚠️ skfuzzy non installé, Fuzzy skipped")
        temps['Fuzzy'] = None

    # # --- Louvain ou autre méthode réseau
    # try:
    #     import networkx as nx
    #     import community as community_louvain
    #     print("▶ Louvain...")
    #     t0 = time.time()
    #     df_louvain = clustering_reseau_multi(df_fil, seuil=seuil_corr,
    #                                          cle_geo=cle_geo, cle_ID=cle_ID, col_val=col_val)
    #     df_final = df_final.merge(df_louvain, on=cle_ID, how='left')
    #     temps['Louvain'] = round(time.time() - t0, 3)
    # except ModuleNotFoundError:
    #     print("⚠️ networkx/python-louvain non installé, Louvain skipped")
    #     temps['Louvain'] = None

    # --- Autres méthodes (Ward/GMM/LDA) peuvent rester “hard” ou être adaptées au soft clustering

    # Affichage temps
    print("\n⏱ Temps de calcul par méthode (s) :")
    for method, t in temps.items():
        if t is not None:
            print(f" - {method:<8}: {t:.3f} s")
        else:
            print(f" - {method:<8}: skipped")

    return df_final, temps


In [ ]:
df_communautes, temps = former_communautes_all_multi(
    df_fil=df_filt,
    cle_geo=cle_geo,
    cle_ID=cle_ID,
    col_val=col_val,
    n_clusters=50,        # nombre de communautés
    m_fuzzy=2,           # paramètre Fuzzy
    seuil_fuzzy=0.2     # seuil d’appartenance pour plusieurs clusters par espèce
)

# Vérifier combien de clusters une espèce peut avoir
fuzzy_cols = [c for c in df_communautes.columns if 'Fuzzy' in c]
df_communautes['n_clusters_Fuzzy'] = df_communautes[fuzzy_cols].sum(axis=1)
df_communautes[[cle_ID, 'n_clusters_Fuzzy']].head()


### Recherche d'une espèce

In [ ]:
# ---------------------------------------------------------------
# 5. Recherche et affichage des espèces du cluster d'une espèce donnée
# ---------------------------------------------------------------
num_cluster=None
#ou 
espece = 'Quercus ilex'

save_fig = False
# Recherche du cluster contenant l'espèce
if espece is not None:
    cle_sujet = df_global[df_global['species'] == espece][cle_ID].unique()[0]
    num_cluster = chercher_numcluster_espece(df_especes_cluster, cle_ID, cle_sujet)
    
if num_cluster is not None:
    df_global_cluster['Cluster_corr'] = pd.to_numeric(df_global_cluster['Cluster_corr'], errors='coerce').astype('Int64')
else:
    print("Saisir un numéro de cluster ou une espèce")

# Liste des espèces du cluster sélectionné, triée selon col_valeur
liste_especes_cluster_choisi = lister_especes_dans_cluster(df_global_cluster, num_cluster,colonnes_obs,col_valeur=col_val, cle_ID=cle_ID)

print(f'Le cluster regroupe {len(liste_especes_cluster_choisi)} espèces')

# Affichage du cluster
afficher_dataframe(liste_especes_cluster_choisi, [col_val] + liste_col_taxo, col_sort=col_val, n_rows=20)


In [ ]:
# Liste des espèces par cluster
especes_par_cluster = df_especes_cluster.groupby('Cluster_corr')['vernacularName_fr'].unique()

# Pour afficher :
for cluster, especes in especes_par_cluster.items():
    print(f"\nCluster {cluster} :")
    for espece in especes:
        print(f" - {espece}")


In [ ]:

colormap = 'viridis'
fond_de_carte = "France"
col_val = 'nombreObs_unique'
titre = f'Drosera rotundifolia - {col_val} - Cluster numéro {num_cluster}'

df_filt = df_global_cluster[df_global_cluster['Cluster_corr'] == num_cluster]

quantile_inf = 0.0
#df_filt = df_filt[df_filt[col_val] > df_filt[col_val].quantile(quantile_inf, interpolation='linear')]

# Configuration de la carte
fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='OpenStreetMap')

# Ajout des couches cartographiques
fig, ax = ajouter_couche_continue(fig, ax, df_filt, carte_maille, col_val, cle_geo,
                                  quantile_inf=0.0, quantile_sup=1,
                                  cmap_choice=colormap, val_alpha=0.6, colorbar_choice=True)

fig, ax = ajouter_couche_SIG(fig, ax, border_local_geo, linewidth=1, edgecolor='grey', linestyle='--')

# Ajout du titre et des annotations
fig.text(0.5, 0.09, f'col_corr : {col_val}', ha='center', va='center', fontsize=10)
ax.set_title(titre, fontsize=16)

if save_fig:
    # Sauvegarde de la figure
    nom_fichier = f"Carte_{zone_name_short}_clusternum{num_cluster}.png"
    nom_fichier = nom_fichier.replace(" ", "_")
    fig.savefig(f'{save_path}/{nom_fichier}.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# ---------------------------------------------------------------
# 6. Affichage de l'aire de répartition du cluster sélectionné
# ---------------------------------------------------------------
liste_clusters = sorted(df_global_cluster['Cluster_corr'].unique())

for num_cluster in liste_clusters:
    
    colormap = 'viridis'
    fond_de_carte = "France"
    col_val = 'nombreObs_unique'
    titre = f'Oiseaux - {col_val} - Cluster numéro {num_cluster}'
    
    df_filt = df_global_cluster[df_global_cluster['Cluster_corr'] == num_cluster]
    
    quantile_inf = 0.0
    #df_filt = df_filt[df_filt[col_val] > df_filt[col_val].quantile(quantile_inf, interpolation='linear')]
    
    # Configuration de la carte
    fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='OpenStreetMap')
    
    # Ajout des couches cartographiques
    fig, ax = ajouter_couche_continue(fig, ax, df_filt, carte_maille, col_val, cle_geo,
                                      quantile_inf=0.0, quantile_sup=0.98,
                                      cmap_choice=colormap, val_alpha=0.75, colorbar_choice=True)
    
    fig, ax = ajouter_couche_SIG(fig, ax, border_local_geo, linewidth=1, edgecolor='grey', linestyle='--')
    
    # Ajout du titre et des annotations
    fig.text(0.5, 0.09, f'col_corr : {col_val}', ha='center', va='center', fontsize=10)
    ax.set_title(titre, fontsize=16)
    
    if save_fig:
        # Sauvegarde de la figure
        nom_fichier = f"Carte_{zone_name_short}_clusternum{num_cluster}.png"
        nom_fichier = nom_fichier.replace(" ", "_")
        fig.savefig(f'{save_path}/{nom_fichier}.png', dpi=300, bbox_inches='tight')
    
    plt.show()


In [ ]:
# 1. Dictionnaire de noms de clusters
nom_clusters = {
    1: "Bassin Parisien",
    2: "Provence",
    3: "Montagne - Vosges",
    4: "JSP1",
    5: "JSP2"

}

In [ ]:
# 2. Calculer les effectifs
df_nb_especes_par_famille = (
    df_especes_cluster
    .groupby(['Cluster_corr', 'family'])['species']
    .nunique()
    .reset_index(name='nb_especes')
)

# 3. Remplacer les numéros de cluster par les noms
df_nb_especes_par_famille['nom_cluster'] = df_nb_especes_par_famille['Cluster_corr'].map(nom_clusters)

# 4. Calculer le total d'espèces par famille sur tous les clusters
total_par_famille = df_nb_especes_par_famille.groupby('family')['nb_especes'].sum()

# 5. Garder les 10 familles avec le plus d'espèces
top_10_familles = total_par_famille.sort_values(ascending=False).head(100).index

# 6. Pivot pour préparer le graphique (filtrer avant pivot pour ne garder que top 10)
pivot_df = (
    df_nb_especes_par_famille[df_nb_especes_par_famille['family'].isin(top_10_familles)]
    .pivot(index='family', columns='nom_cluster', values='nb_especes')
    .fillna(0)
)

# 7. Tracer le graphique
pivot_df.plot(kind='bar', figsize=(14, 6))

plt.title("Nombre d'espèces par famille et par cluster (Top 10 familles)")
plt.xlabel("Famille")
plt.ylabel("Nombre d'espèces")
plt.xticks(rotation=45, ha='right')
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()


In [ ]:


# 2. Calculer les effectifs
df_nb_especes_par_famille = (
    df_especes_cluster
    .groupby(['Cluster_corr', 'family'])['species']
    .nunique()
    .reset_index(name='nb_especes')
)

# 3. Remplacer les numéros de cluster par les noms
df_nb_especes_par_famille['nom_cluster'] = df_nb_especes_par_famille['Cluster_corr'].map(nom_clusters)

# 4. Pivot pour avoir les clusters en X, familles en légende
pivot_df = df_nb_especes_par_famille.pivot(index='nom_cluster', columns='family', values='nb_especes').fillna(0)

# 5. Tracer
pivot_df.plot(kind='bar', figsize=(12, 6))

plt.title("Nombre d'espèces par cluster (biome), par Famille")
plt.xlabel("Biome")
plt.ylabel("Nombre d'espèces")
plt.xticks(rotation=0)
plt.legend(title="Famille", bbox_to_anchor=(1.02, 1), loc='upper left')  # Légende à droite
plt.tight_layout()
plt.show()


### Etude d'une sous-zone

In [19]:
RNR_gpd

,objectid,id_local,nom_site,date_crea,url_fiche,surf_off,acte_deb,operateur,id_mnhn,code_dep,...,densdr,surface,perimetre,nb_commune,st_area_shape_2,st_length_shape_2,st_area_shape,st_perimeter_shape,geo_point_2d,geometry
0,1,RNR88,Scamandre,2006-11-29 01:00:00+00:00,http://inpn.mnhn.fr/espace/protege/FR9300033,146.70,FR930003320061129,Réserves naturelles de France,FR9300033,30,...,124.79,"5,875.56",631.38,353,"1,485,919.74",0.00,"1,485,919.74","6,405.03","{ ""lon"": 4.3401467612709297, ""lat"": 43.6113491...","POLYGON ((4.35104 43.60541, 4.35105 43.60518, ..."
1,9,RNR226,Marais de Bonnefont,2011-02-10 01:00:00+00:00,http://inpn.mnhn.fr/espace/protege/FR9300095,42.00,FR930009520110210,Réserves naturelles de France,FR9300095,46,...,33.27,"5,221.95",469.90,326,"422,884.21",0.00,"422,884.21","3,540.41","{ ""lon"": 1.7947092444623627, ""lat"": 44.8194518...","POLYGON ((1.79391 44.82404, 1.79397 44.82384, ..."
2,6,RNR157,Gorges du Gardon,2009-12-18 01:00:00+00:00,http://inpn.mnhn.fr/espace/protege/FR9300037,491.34,FR930003720091218,Réserves naturelles de France,FR9300037,30,...,124.79,"5,875.56",631.38,353,"4,983,261.33",0.00,"4,983,261.33","45,766.98","{ ""lon"": 4.412135354610089, ""lat"": 43.94086772...","MULTIPOLYGON (((4.43368 43.94977, 4.43377 43.9..."
3,2,RNR85,Combe Chaude,2006-12-21 01:00:00+00:00,http://inpn.mnhn.fr/espace/protege/FR9300034,56.28,FR930003420061221,Réserves naturelles de France,FR9300034,30,...,124.79,"5,875.56",631.38,353,"553,165.09",0.00,"553,165.09","5,530.73","{ ""lon"": 3.721215284221052, ""lat"": 43.96710875...","POLYGON ((3.73282 43.97236, 3.73281 43.9718, 3..."
4,5,RNR202,Sainte Lucie,2009-09-25 02:00:00+00:00,http://inpn.mnhn.fr/espace/protege/FR9300036,825.00,FR930003620090925,Réserves naturelles de France,FR9300036,11,...,57.42,"6,355.00",545.84,436,"8,211,714.55",0.00,"8,211,714.55","22,326.11","{ ""lon"": 3.0589907420582385, ""lat"": 43.0455598...","MULTIPOLYGON (((3.07698 43.05912, 3.0771 43.05..."
5,10,RNR266,Mahistre et Musette,2013-02-04 01:00:00+00:00,http://inpn.mnhn.fr/espace/protege/FR9300137,260.57,FR930013720130204,Réserves naturelles de France,FR9300137,30,...,124.79,"5,875.56",631.38,353,"2,660,461.81",0.00,"2,660,461.81","10,852.39","{ ""lon"": 4.2329953154788322, ""lat"": 43.5996744...","MULTIPOLYGON (((4.23879 43.59323, 4.23751 43.5..."
6,8,RNR227,Coteaux du Fel,2011-02-10 01:00:00+00:00,http://inpn.mnhn.fr/espace/protege/FR9300094,80.75,FR930009420110210,Réserves naturelles de France,FR9300094,12,...,31.67,"8,771.00",628.48,286,"835,288.80",0.00,"1,266,028.70","17,931.47","{ ""lon"": 2.522073654246201, ""lat"": 44.65644463...","MULTIPOLYGON (((2.52338 44.6638, 2.52338 44.66..."
7,4,RNR128,Nyer,2007-10-18 02:00:00+00:00,http://inpn.mnhn.fr/espace/protege/FR9300035,"2,192.34",FR930003520071018,Réserves naturelles de France,FR9300035,66,...,111.49,"4,150.33",425.46,226,"23,467,752.82",0.00,"23,467,752.82","33,373.60","{ ""lon"": 2.2759967262897374, ""lat"": 42.4922895...","POLYGON ((2.27601 42.5315, 2.27605 42.53116, 2..."
8,13,RNR309,Massif de Saint-Barthélemy,2015-11-16 01:00:00+00:00,https://inpn.mnhn.fr/espace/protege/FR9300160,460.77,None,Réserves naturelles de France,FR9300160,09,...,NaN,NaN,NaN,1,"7,440,868.86",0.00,"7,440,868.86","12,453.03","{ ""lon"": 1.8509311956504215, ""lat"": 42.8363783...","POLYGON ((1.85975 42.82638, 1.85963 42.82577, ..."
9,3,RNR272,Cambounet-sur-le-Sor,2013-11-29 01:00:00+00:00,http://inpn.mnhn.fr/espace/protege/FR9300131,30.87,FR930013120131129,Réserves naturelles de France,FR9300131,81,...,66.03,"5,783.85",492.11,320,"313,979.96",0.00,"313,979.96","2,972.93","{ ""lon"": 2.1349033931130141, ""lat"": 43.5833236...","POLYGON ((2.13536 43.57735, 2.13526 43.5773, 2..."


In [ ]:
# ---------------------------------------------------------------
# 7. Sélection des codes de maille pour un site donné
# ---------------------------------------------------------------
parc_gpd = PN_et_PNR_gpd
nom_parc = 'Ballons des Vosges'
cle_nom_site = 'NOM_SITE'
methode_nom = 'contains' # contains exact
taux_min=0.6 # Taux de la maille qui doit être compris dans la zone : 1= 100% de la maille dans la zone

liste_codes = lister_mailles_dans_site(carte_maille, parc_gpd, nom_parc, taux_min=taux_min, cle_geo=cle_geo,
                                       cle_nom_site=cle_nom_site, methode=methode_nom)
df_global_raw=df_global.copy()
print(f"Nombre de mailles dans la zone choisie : {len(liste_codes)}")

In [ ]:
# ---------------------------------------------------------------
# 8. Filtrer les mailles présentes dans liste_codes et grouper par cluster
# ---------------------------------------------------------------
col_choice = 'nombreObs_norm_par_espece' 
crit='sum'
nmin=200 #Si on veut filtrer les espèces les + présentes
crit_norm=True


df_global=filtrer_top_global(df_global_raw,var='nombreObs',nmin=nmin,cle_ID=cle_ID)
df_global_cluster = pd.merge(df_global, df_especes_cluster[[cle_ID, 'Cluster_corr']], on=cle_ID)
df_global_cluster['Cluster_corr'] = pd.to_numeric(df_global_cluster['Cluster_corr'], errors='coerce').astype('Int64')


df_local = df_global[df_global[cle_geo].isin(liste_codes)]
df_grouper_par_cluster_local = grouper_par_cluster(df_local, df_especes_cluster, colonnes_obs,col_choice, cle_geo=cle_geo, cle_ID=cle_ID,crit=crit,norm=crit_norm)
afficher_dataframe(df_grouper_par_cluster_local, ['Cluster_corr', 'species', 'vernacularName_fr',
                                                  'nombreObs', 'nombreObs_norm_par_espece',
                                                  'nombreObs_norm_par_maille_et_kingdom'],
                   col_sort=col_choice).head(10)


In [ ]:
# ---------------------------------------------------------------
# 9. Afficher les détails du cluster choisi dans la zone 
# ---------------------------------------------------------------

# Paramétrage
rank_cluster = 0
col_abondance = 'nombreObs'  # Pour les abondances brutes
col_critere = 'nombreObs_norm_par_espece'  # Pour le critère de tri

# Récupération du numéro du cluster sélectionné
num_cluster = int(df_grouper_par_cluster_local['Cluster_corr'].iloc[rank_cluster])
print(f'Cluster n° {num_cluster}')

# Préparation du DataFrame local avec l'information de cluster
df_local_cluster = pd.merge(df_local, df_especes_cluster[[cle_ID, 'Cluster_corr']], on=cle_ID)
df_local_cluster['Cluster_corr'] = pd.to_numeric(df_local_cluster['Cluster_corr'], errors='coerce').astype('Int64')

# Étude du cluster local via la fonction
resultat = etudier_un_cluster_local(
    df_global_cluster, 
    df_local_cluster, 
    cle_ID=cle_ID, 
    num_cluster=num_cluster,
    colonnes_obs=colonnes_obs,
    col_choice_1=col_abondance,
    col_choice_2=col_critere
)

# Affichage du résultat (en dehors de la fonction comme tu l'as voulu)
print(f'Le cluster regroupe {len(resultat)} espèces')
afficher_dataframe(resultat, [col_abondance, col_critere] + liste_col_taxo, col_sort=col_critere, n_rows=10)


In [ ]:
# ---------------------------------------------------------------
# 9. Afficher les espèces absentes dans la zone étudiée mais les plus susceptibles d'y être présentes
# ---------------------------------------------------------------
max_cluster=df_local_cluster['Cluster_corr'].nunique()

resultat=rechercher_especes_localement_absentes(df_global_cluster, df_local_cluster,df_grouper_par_cluster_local, cle_ID,colonnes_obs, col_choice='nombreObs',seuil=None,max_cluster=min(max_cluster,50))

afficher_dataframe(resultat, ['num_cluster'] + liste_col_taxo,n_rows=10)

## ANALYSE PAR CORRELATION

### Entre un sujet et un objet

In [61]:
# Définition des paramètres
clade = 'species' 
sujet = 'Colchicum filifolium'
col_corr = 'nombreObs_norm_par_maille_et_kingdom_log'
methode_choice = 'pearson'  # Options: 'pearson', 'kendall', 'spearman'
col_filt = 'nombreObs_norm_par_maille_et_kingdom'
obs_min=1000
filtre=None
seuil_observation = 5 #nombre minimum d'obs par maille du sujet pour être pris en compte

# Filtrage et préparation des données
df_filt=df_biodiv.copy()
df_filt = df_filt[
    (df_filt["kingdom"] == "Plantae")
]
df_sujet = df_filt[(df_filt[clade] == sujet) & (df_filt['nombreObs'] >= seuil_observation)]
#df_filt = df_filt[(df_filt['nombreObs'] >= seuil_observation)]
if filtre is not None:
    df_filt = filtrer_categorie(df_filt, filtre,filtres)
df_global = filtrer_top_global(df_filt, col_filt, obs_min, cle_ID)

if df_sujet.empty:
    raise ValueError(f"No data found for {sujet} in {clade}.")

# Combinaison et nettoyage des données
df_global = pd.concat([df_sujet, df_global]).drop_duplicates()
df_global_complet = completer_df(df_global, df_global, cle_geo, cle_ID)
df_global_complet = df_global_complet.sort_values(by=cle_ID)

# Calcul des corrélations avec mesure du temps
t_start = time.time()
df_corr = calculer_correlation_sujet(
    df_global_complet, col_corr, clade, sujet, 
    methode=methode_choice, cle_ID=cle_ID, cle_geo=cle_geo
)
t_end = time.time()
print(f"Temps d'exécution : {t_end - t_start:.2f} secondes")

# Affichage des résultats
afficher_dataframe(df_corr, ['Coeff_corr'] + liste_col_taxo, col_sort='Coeff_corr').head(10)

[global] 2189 espèces retenues (28% du total)
Temps d'exécution : 32.65 secondes


,Coeff_corr,speciesKey,species,vernacularName_fr,vernacularName_en,genus,family,order,class,phylum,kingdom,occurrenceID
0,1.00,2739964,Colchicum filifolium,NaN,NaN,Colchicum,Colchicaceae,Liliales,Liliopsida,Tracheophyta,Plantae,http://flore.silene.eu/occtax/ddb8a992-d8a4-4a...
1,0.28,7331330,Helianthemum marifolium,NaN,NaN,Helianthemum,Cistaceae,Malvales,Magnoliopsida,Tracheophyta,Plantae,http://flore.silene.eu/occtax/25a873a8-2ca0-47...
2,0.22,2977292,Ononis mitissima,NaN,Mediterranean Restharrow,Ononis,Fabaceae,Fabales,Magnoliopsida,Tracheophyta,Plantae,22228f3a-b0f7-4774-ae33-4c66eabffda1
3,0.18,2792833,Ophrys bertolonii,NaN,Bertoloni'S Bee Orchid,Ophrys,Orchidaceae,Asparagales,Liliopsida,Tracheophyta,Plantae,L2015.3.1426
4,0.15,2857323,Allium chamaemoly,NaN,NaN,Allium,Amaryllidaceae,Asparagales,Liliopsida,Tracheophyta,Plantae,http://flore.silene.eu/occtax/102b06b4-09e2-4e...
5,0.15,7419174,Limonium duriusculum,NaN,European Sea Lavendar,Limonium,Plumbaginaceae,Caryophyllales,Magnoliopsida,Tracheophyta,Plantae,415B9F61-2679-6BC4-E053-2614A8C08B7C
6,0.15,2965379,Medicago arborea,Luzerne Arborescente,Tree Medick,Medicago,Fabaceae,Fabales,Magnoliopsida,Tracheophyta,Plantae,554e2dcd-8525-4f6f-a7a7-327cb0cc6d50
7,0.12,5348659,Coronilla juncea,NaN,NaN,Coronilla,Fabaceae,Fabales,Magnoliopsida,Tracheophyta,Plantae,http://flore.silene.eu/occtax/57fee0c9-e6b2-48...
8,0.11,5403742,Scolymus hispanicus,Scolyme D'Espagne,Golden Thistle,Scolymus,Asteraceae,Asterales,Magnoliopsida,Tracheophyta,Plantae,b39b260c-7750-448c-b587-baeb63dd5799
9,0.11,3088761,Limbarda crithmoides,NaN,Golden Samphire,Limbarda,Asteraceae,Asterales,Magnoliopsida,Tracheophyta,Plantae,b045c233-77b9-4057-8548-c20833cf5cec


In [ ]:
# Filtrer les résultats
df_filt = filtrer_categorie(df_corr, 'plantes',filtres)
afficher_dataframe(df_filt,['Coeff_corr']+liste_col_taxo,col_sort='Coeff_corr').head(10)

In [ ]:
df_filt[df_filt['species']==sujet]

In [63]:
# Paramètres de l'affichage
colormap = 'plasma'
SIG_defaut = True
save_carte=False
log_values=False
affichage_zone_recensee=True

with_specie = 1  # 1 : prend en compte l'espèce, 0 : ne la prend pas en compte

n_sigma = 2  # 0 : seuil = 0
seuil_quantile = 0.1
#seuil_observation = 1
seuil_prediction=0

# Identification de l'espèce cible
cle_sujet = df_biodiv[df_biodiv['species'] == sujet][cle_ID].unique()[0]

# Calcul de la prédiction
col_recalcul = col_corr
colonne_resultat = f"{col_recalcul}_predit"
df_sujet_predit = recalculer_nombreObs_par_correlation(
    df_global, df_corr, col_recalcul, cle_sujet, with_specie,
    cle_ID=cle_ID, cle_geo=cle_geo
)

# Définition du titre
titre = f"Aire de répartition potentielle de {sujet}"

# Détermination du seuil de prédiction
df_global_avec_presence = df_global[
    (df_global['species'] == sujet) & (df_global['nombreObs'] >= seuil_observation)
]
liste_mailles_avec_sujet = df_global_avec_presence[cle_geo].unique()
df_predit_avec_presence = df_sujet_predit[df_sujet_predit[cle_geo].isin(liste_mailles_avec_sujet)]
if seuil_quantile != 0:
    seuil_prediction = df_predit_avec_presence[colonne_resultat].quantile(seuil_quantile)
if n_sigma != 0:
    seuil_prediction=calculer_seuil(df_global,df_sujet_predit,cle_sujet,col_corr,n_sigma=n_sigma,cle_geo=cle_geo,cle_ID=cle_ID)
df_filt = df_sujet_predit[df_sujet_predit[colonne_resultat] >= seuil_prediction]

# Affichage du fond de carte
fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='Esri')

# Ajout des données prédites
fig, ax = ajouter_couche_continue(
    fig, ax, df_filt, carte_maille, colonne_resultat, cle_geo,
    quantile_inf=0.0, quantile_sup=1,
    borne_min=None, borne_max=None,
    cmap_choice=colormap, val_alpha=0.8,
    log_values=log_values
)

# Ajout des couches SIG par défaut si activé
if SIG_defaut:
    fig, ax = charger_couches_SIG_defaut(fig, ax)

# Ajout des points de présence réelle
if affichage_zone_recensee:
    df_global_avec_presence = df_global[(df_global['species'] == sujet) & (df_global['nombreObs'] >= seuil_observation)]
    fig, ax = ajouter_couche_point(
        fig, ax, df_global_avec_presence, carte_maille,
        col_valeur='nombreObs', cle_geo=cle_geo,
        color_dot='black', size_dot=2, legend_choice=True
    )

# Ajout du titre et informations complémentaires
ax.set_title(titre, fontsize=16)
fig.text(0.5, 0.09, 
         f'Méthode de corr : {methode_choice}, col_corr : {col_corr}, col_recalcul : {col_recalcul}, with_specie : {with_specie}, sigma={n_sigma}',
         ha='center', va='center', fontsize=10)

# Sauvegarde de la carte
if save_carte:
    fig.savefig(f"{save_path}/{titre}.png", dpi=300, bbox_inches='tight')

# Affichage de la carte
plt.show()

In [ ]:
# Affichage et regression des données prédites en fonction des données d'observation
col_values_corr='nombreObs_norm_par_maille_et_kingdom'
colonne_predit = f"{col_values_corr}_predit"
x, y = prepare_data(df_global, df_sujet_predit, cle_geo, sujet, col_values_corr)
coefficients = fit_and_plot(x, y, col_values_corr, sujet, loi='all')
plot_residuals(x, y, coefficients['linear'][0], coefficients['linear'][1])


### Extrapolation pour toutes les espèces

In [118]:
# Paramètres 
method="combined"

var_global='nombreObs'
nmin_filt_predic= 10

var_mailles = 'nombreObs'
ntop_filt_predic=10

seuil_observation = 2 
filtre="plantes"

df_global=df_biodiv.copy()
df_global = df_global[(df_global['nombreObs'] >= seuil_observation)]
    
if filtre is not None:
    df_global = filtrer_categorie(df_global, filtre,filtres)
    
if method is not None:
    df_global= filtrer_top(df_global,method, 
                              var_global=var_global,var_mailles=var_mailles,
                              ntop_mailles=ntop_filt_predic,nmin_global=nmin_filt_predic, 
                              cle_ID=cle_ID,cle_geo=cle_geo)
    
print(f"Nombre d'espèces prises en compte : {len(df_global['speciesKey'].unique())}")


[combined] 3198 espèces retenues (79% du total)
Nombre d'espèces prises en compte : 3198


In [152]:
# ---------------------------------------------------------------
# 1. Filtrage des espèces pour réduire le temps de calcul
# ---------------------------------------------------------------

col_val = 'nombreObs_norm_par_maille_et_kingdom'
methode_corr = 'kendall'  # pearson, kendall
import_mat_corr = False
calculate_mat_corr = True
save_mat_corr = True

path_data_mat_corr = os.path.join(data_path, "mat_corr")
filename_mat_corr=f'mat_corr_{zone_name_short}_{"".join(regnes_a_garder)}_{nmin_filt_predic}-{ntop_filt_predic}_{col_val}_{methode_corr}_{cle_geo}.csv'

# ---------------------------------------------------------------
# 2. Calcul ou importation de la matrice de corrélation
# ---------------------------------------------------------------
if import_mat_corr:
    # Chargement des données
    mat_corr = pd.read_csv(os.path.join(path_data_mat_corr, filename_mat_corr), dtype=str)
    print("✅ Importation terminée")

    
elif calculate_mat_corr:
    # Jalon 1 : début du processus
    start_time = time.time()
    mat_corr = calculer_matrice_correlation(df_global, col_val, methode_corr, cle_geo, cle_ID)
    mat_corr = mat_corr.astype(float)
    end_time = time.time()
    
    print(f"Temps écoulé: {end_time - start_time:.2f} secondes")
    print("✅ Calcul terminé")

if save_mat_corr:
    # Sauvegarde de la matrice de corrélation
    mat_corr_tosave = pd.DataFrame(mat_corr)
    mat_corr_tosave.to_csv(os.path.join(path_data_mat_corr, filename_mat_corr), index=False)
    print("✅ Sauvegarde terminée")


Temps écoulé: 1053.49 secondes
✅ Calcul terminé
✅ Sauvegarde terminée


Pearson :
3127 espèces, 4495 mailles -> 96 s
4379 espèces, 1506 mailles -> 59 s
6000 espèces, 324 mailles -> 12 s

kendall :
3044 espèces 8000 mailles -> 4,268 s
4453 espèces, 1507 mailles -> 2,451 s
2663  espèces, 324 mailles -> 
3198 espèces, 874 mailles -> 1053 s

In [153]:
# ---------------------------------------------------------------
# 3. Complétion du DataFrame et calcul de la prédiction
# ---------------------------------------------------------------
import_df_predit = False
by_chunk=False

calculate_df_predit=True

save_df_predit = True
col_val_predit=f"{col_val}_predit"

path_data_df_predit = os.path.join(data_path, "df_predit")
filename_df_predit=f'df_predit_{zone_name_short}_{"".join(regnes_a_garder)}_{nmin_filt_predic}-{ntop_filt_predic}_{col_val}_{methode_corr}_{cle_geo}.csv'

df_complet_predit=[]
if import_df_predit:
    if by_chunk:
        chunk_size = 100000  # Nombre de lignes par chunk
        chunks = pd.read_csv(os.path.join(path_data_df_predit, filename_df_predit),chunksize=chunk_size,low_memory=False)
        df_complet_predit = pd.concat(chunks, ignore_index=True)
    else:
        df_complet_predit = pd.read_csv(os.path.join(path_data_df_predit, filename_df_predit),low_memory=False)
    print("✅ Importation terminée")

elif calculate_df_predit:
    print("📥 Calcul en cours...")
    
    # Complétion du DataFrame
    liste_codes_complet = df_global[cle_geo].unique()
    df_global_complet = completer_df(df_global, df_global, cle_geo, cle_ID)
    df_global_complet = df_global_complet.sort_values(by=cle_ID, ascending=True)

    # Jalon 2 : début du processus de prédiction
    start_time = time.time()
    df_complet_predit = calculer_prediction(df_global_complet, mat_corr, col_val, cle_geo, cle_ID)

    df_biodiv_complet = df_biodiv.drop(columns=[col_val_predit], errors='ignore')
    df_biodiv_complet = pd.merge(df_biodiv_complet, df_complet_predit[[cle_geo, cle_ID, col_val_predit]], on=[cle_geo, cle_ID])

    
    # Calculer et afficher le temps écoulé
    end_time = time.time()
    print(f"Temps écoulé: {end_time - start_time:.2f} secondes")

    # Normalisation par espèce
    df_complet_predit = normaliser_par_espece(df_complet_predit, cle_ID, col_val_predit)
    df_complet_predit = normaliser_log(df_complet_predit, col_val_predit)

    print("✅ Calcul et normalisations terminés")
    
if save_df_predit:
    print("📥 Sauvegarde en cours...")
    # Sauvegarde des prédictions
    path_data_predit = os.path.join(data_path, "df_predit")
    df_tosave = pd.DataFrame(df_complet_predit)
    df_tosave.to_csv(os.path.join(path_data_df_predit, filename_df_predit), index=False) 
    print("✅ Sauvegarde terminée")


📥 Calcul en cours...
Nombre total de mailles: 874
Progress: 1% | Temps total estimé : 167.2s | Heure de fin estimée : 00:01:49
Temps écoulé: 164.67 secondes
Le nombre d observations total par espèce est fixé à 10 000
✅ Calcul et normalisations terminés
📥 Sauvegarde en cours...
✅ Sauvegarde terminée


5800 mailles, 944 especes ->  3500 s (1h)
5800 mailles, 6606 especes ->  25_800s (7h)
313 mailles, 2556 espèces -> 39 s
4369 mailles, 3127 espèces,  -> 6700 (2h)


In [ ]:
# ---------------------------------------------------------------
# 4. Configuration de la carte
# ---------------------------------------------------------------
# Sélection de l'espèce
espece = 'Colchicum filifolium'
cle_sujet = df_biodiv[df_biodiv['species'] == espece][cle_ID].unique()[0]
col_values_corr = 'nombreObs_norm_par_maille_et_kingdom'
colonne_predit = f"{col_values_corr}_predit"

#fond_de_carte = "Italie"
colormap = 'plasma'
n_sigma = 1 # 1 pour animal, 2 pour plantes
seuil_observation = 1
min_val_predit=None
zoom_size = 7
save = True

titre = f'Aire de répartition potentielle de {espece}'

# Filtrage des données
df_complet_predit_espece = df_complet_predit[df_complet_predit['species'] == espece]
df_global_avec_presence = df_global[(df_global['species'] == espece) & (df_global['nombreObs'] >= seuil_observation)]
liste_mailles_avec_espece = df_global_avec_presence[cle_geo].unique()
df_complet_predit_avec_presence = df_complet_predit_espece[df_complet_predit_espece[cle_geo].isin(liste_mailles_avec_espece)]

# Calcul du seuil de prédiction
seuil_prediction = calculer_seuil(
    df_global, df_complet_predit_espece, cle_sujet, col_values_corr, seuil_observation,
    n_sigma, cle_ID=cle_ID, cle_geo=cle_geo
)
if min_val_predit:
    seuil_prediction=min_val_predit

df_filt = df_complet_predit_espece[df_complet_predit_espece[colonne_predit] >= seuil_prediction]

# ---------------------------------------------------------------
# 5. Affichage de la carte
# ---------------------------------------------------------------
fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='OpenStreetMap')

# Ajout des données
fig, ax = ajouter_couche_continue(
    fig, ax, df_filt, carte_maille, colonne_predit, cle_geo,
    quantile_inf=0.01, quantile_sup=0.99,
    cmap_choice=colormap, val_alpha=0.75, colorbar_choice=True
)

# Ajout des points de présence
fig, ax = ajouter_couche_point(
    fig, ax, df_global_avec_presence, carte_maille, col_valeur='nombreObs',
    cle_geo=cle_geo, color_dot='black', size_dot=1, legend_choice=True
)

# Ajout de couches SIG
#fig, ax = ajouter_couche_SIG(fig, ax, departement_gpd, linewidth=1, edgecolor='grey', linestyle='--')

# Ajout du titre et des annotations
ax.set_title(titre, fontsize=16)
fig.text(
    0.5, 0.09, 
    f'Méthode de corr : {methode_corr}, col_corr : {col_values_corr}, col_recalcul : {col_values_corr}, sigma={n_sigma}',
    ha='center', va='center', fontsize=10
)

# Sauvegarde et affichage
if save:
    fig.savefig(f"{save_path}/{titre}.png", dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
# ---------------------------------------------------------------
# 6. Regressions des données prédites en fonction des données d'observation
# ---------------------------------------------------------------
col_values_corr='nombreObs_norm_par_maille_et_kingdom'
colonne_predit = f"{col_values_corr}_predit"
x, y = prepare_data(df_global, df_complet_predit_espece, cle_geo, espece, col_values_corr)
coefficients = fit_and_plot(x, y, col_values_corr, espece, loi='all')
plot_residuals(x, y, coefficients['linear'][0], coefficients['linear'][1])


In [ ]:
# ---------------------------------------------------------------
# 7. Conversion en données quantitatives et affichage de la carte pour le taxon choisi
# ---------------------------------------------------------------

df_filt=df_complet_predit_espece.copy()
methode='linear'

colonne_predit_modelisation='nombreObs_predit_glm_'+methode
df_filt_trans = appliquer_transformation(df_filt, colonne_predit, methode, coefficients)
df_filt_trans = df_filt_trans.dropna(subset=[colonne_predit_modelisation])  # Remove NaNs in the column
df_filt_trans=df_filt_trans[df_filt_trans[colonne_predit_modelisation]>=0]
#fig, ax=configurer_carte('OpenStreetMap',center_x,center_y,height,zoom=zoom_size,fig_size=(size_x, size_y))
fig, ax = afficher_fond_carte(zone_name_short, dictionnaire_cartes, source_fond='OpenStreetMap')
fig, ax=ajouter_couche_continue(fig, ax, df_filt_trans,carte_maille,colonne_predit_modelisation,cle_geo,quantile_inf=0.0,quantile_sup=1,
                                cmap_choice=colormap,val_alpha=0.75,colorbar_choice=True)
#fig, ax=ajouter_couche_point(fig, ax,df_global_avec_presence,carte_maille,col_valeur='nombreObs',cle_geo=cle_geo,color_dot='black',size_dot=2,legend_choice=True)
fig, ax=ajouter_couche_SIG(fig, ax,border_local_geo,linewidth=1,edgecolor='grey',linestyle='--')
fig.text(0.5, 0.09, f'Méthode de corr : {methode}, col_corr : {col_values_corr}, col_recalcul : {col_values_corr}, sigma={n_sigma}',
         ha='center', va='center', fontsize=10)
#fig, ax=ajouter_couche_SIG(fig, ax,departement_gpd)
titre = f'Aire de répartition prédite de {espece} - modèle {methode}'
ax.set_title(titre, fontsize=16)  # Taille de la police définie à 16
fig.savefig(save_path+'/'+titre+'.png', dpi=300, bbox_inches='tight')  # Enregistre au format PNG avec une résolution de 300 DPI

plt.show()

### Recherche des espèces potentiellement présentes dans une sous-zone

In [ ]:
# ---------------------------------------------------------------
# 1. Sélection des codes de maille pour un site donné
# ---------------------------------------------------------------
mode='mailles' #zone ou mailles

if mode == 'zone':
    parc_gpd = PN_et_PNR_gpd #geodataframe contenant la geometry de la zone à étudier : PN_et_PNR_gpd departement_gpd
    cle_nom_site = 'NOM_SITE' #nom de la colonne contenant le nom/clé de la zone à étudier : NOM_SITE code
    nom_parc = 'Ballons des Vosges' #nom/clé de la zone à étudier :  Ballons des Vosges 88 
    methode_nom = 'contains' # contains ou exact
    taux_min=1# Taux de la maille qui doit être compris dans la zone : 1= 100% de la maille dans la zone
    liste_codes = lister_mailles_dans_site(carte_maille, parc_gpd, nom_parc, taux_min=taux_min, cle_geo=cle_geo,
                                           cle_nom_site=cle_nom_site, methode=methode_nom)

elif mode=='mailles':
    liste_codes=['10kmE01393N03795ITA']

print(f"Nombre de mailles dans la zone choisie : {len(liste_codes)}")

In [ ]:
# ---------------------------------------------------------------
# 2. Prédiction au niveau local
# ---------------------------------------------------------------
col_values_prediction='nombreObs_norm_par_maille_et_kingdom' # 'nombreObs_norm_par_maille_et_kingdom' 'nombreObs' 'nombreObs_norm_par_espece'

# Complétion du DataFrame
df_local=df_global[(df_global[cle_geo].isin(liste_codes))]
df_local_complet=completer_df(df_local,df_global,cle_geo,cle_ID)
df_local_complet = df_local_complet.sort_values(by=cle_ID,ascending=True)

# Jalon 2 : début du processus de prédiction
df_local_complet_predit=calculer_prediction(df_local_complet, mat_corr, col_values_prediction,cle_geo,cle_ID)
colonne_predit = f"{col_values_corr}_predit"

# Normalisation par espèce
df_local_complet_predit = normaliser_par_espece(df_local_complet_predit, cle_ID, colonne_predit)
df_local_complet_predit = normaliser_log(df_local_complet_predit, colonne_predit)

In [ ]:
# ---------------------------------------------------------------
# 3. Recherche des espèces suceptibles d'être présentes au niveau local
# ---------------------------------------------------------------
filtre='arthropodes'
df_filt = filtrer_categorie(df_local_complet_predit, filtre,filtres)
df_local_especes_absentes=recherche_espece_absente(df_filt,colonne_predit,cle_geo,cle_ID)

print(f"Nombre d'espèces : {df_local_especes_absentes.shape[0]}")

afficher_dataframe(df_local_especes_absentes,[colonne_predit]+liste_col_taxo,col_sort=colonne_predit).head(50)

In [ ]:
nom_espece = "Pamphagus marmoratus"

# Trouver les indices où l'espèce est absente
rang = df_local_especes_absentes.index[df_local_especes_absentes['species'] == nom_espece].tolist()

# Nombre total d'espèces absentes
total = df_local_especes_absentes.shape[0]

if rang:
    print(f"L'espèce '{nom_espece}' est absente à la position {rang[0]} sur {total} espèces.")
else:
    print(f"L'espèce '{nom_espece}' n'est pas présente dans la liste des espèces absentes.")


## SUIVI TEMPOREL

In [ ]:
# Suivi des espèces (ou autres clade) disparues entre période 1 et 2
df_filt=df_biodiv_periode
df_suivi_mailles = suivre_disparition_geo(df_filt,cle_ID,cle_geo)

col_valeur='taux_apparue'
titre=col_valeur
colormap='YlOrRd_r'
fond_carte='GeoportailSatellite'
fig, ax = afficher_fond_carte(zone_name, dictionnaire_cartes, source_fond='OpenStreetMap')
fig, ax=ajouter_couche_continue(fig, ax, df_suivi_mailles,carte_maille,col_valeur,cle_geo,
                                quantile_inf=0.0,quantile_sup=0.99,
                                cmap_choice='viridis',val_alpha=0.7,
                               log_values=False)
fig, ax=ajouter_couche_SIG(fig, ax,border_local_geo,linewidth=1,edgecolor='grey',linestyle='--')
ax.set_title(titre, fontsize=18)  # Taille de la police définie à 16
# Ajoute une ligne de texte en dessous de la figure
fig.text(0.45, 0.15, f'var : {col_valeur}', ha='center', va='center', fontsize=10)
ax.set_title(titre, fontsize=16)  # Taille de la police définie à 16
#fig.savefig(save_path+'/'+titre+'.png', dpi=300, bbox_inches='tight')  # Enregistre au format PNG avec une résolution de 300 DPI
plt.show()

In [ ]:
col_valeur='taux_disparue'
titre=col_valeur
colormap='YlOrRd'
fond_carte='GeoportailSatellite'
fig, ax=configurer_carte('OpenStreetMap',center_x,center_y,height,zoom=zoom_size,fig_size=(size_x, size_y))
fig, ax=ajouter_couche_continue(fig, ax, df_suivi_mailles,carte_maille,col_valeur,cle_geo,
                                quantile_inf=0.0,quantile_sup=0.99,
                                cmap_choice='viridis',val_alpha=0.7,
                               log_values=False)
fig, ax=ajouter_couche_SIG(fig, ax,border_local_geo,linewidth=1,edgecolor='grey',linestyle='--')
ax.set_title(titre, fontsize=18)  # Taille de la police définie à 16
# Ajoute une ligne de texte en dessous de la figure
fig.text(0.45, 0.15, f'var : {col_valeur}', ha='center', va='center', fontsize=10)
ax.set_title(titre, fontsize=16)  # Taille de la police définie à 16
#fig.savefig(save_path+'/'+titre+'.png', dpi=300, bbox_inches='tight')  # Enregistre au format PNG avec une résolution de 300 DPI

plt.show()

In [ ]:
# Afficher la carte de l'évolution de l'aire de répartition d'un taxon
taxon='Arenaria provincialis'
clade='species'
df_filt=df_biodiv_periode[(df_biodiv_periode[clade]==taxon)]
df_filt=df_filt[[cle_geo, clade,'periode']].drop_duplicates().reset_index(drop=True)
grouped = df_filt.groupby([cle_geo, clade])['periode'].apply(list).reset_index()

# Appliquer la fonction pour créer la colonne 'statut'
grouped['statut'] = grouped['periode'].apply(determiner_statut)
# Garder seulement les colonnes nécessaires
final_df = grouped[[cle_geo, clade, 'statut']]

titre=f"Statut de {taxon} en Europe du Sud-Ouest"
            
statut_colors = {
    "Colonisation 1991-2010": "green",
    "Colonisation 2011-2024": "lightgreen",
    "Disparition 1801-1990": "red",
    "Disparition 1991-2011": "orange",
    "Présence 1991-2010 mais absent aujourd'hui": "yellow",
    "Présence continue jusqu'à aujourd'hui": "blue"
}

fig, ax = afficher_fond_carte(zone_name, dictionnaire_cartes, source_fond='OpenStreetMap')
fig, ax=ajouter_couche_statut(fig, ax, final_df, carte_maille, statut_colors,col_valeur='statut',cle_geo=cle_geo, val_alpha=0.75, legend_choice=True,loc_legend="upper right")
fig, ax=ajouter_couche_SIG(fig, ax,border_local_geo,linewidth=1,edgecolor='black',linestyle='-')
#fig, ax=ajouter_couche_SIG(fig, ax,departement_gpd)
ax.set_title(titre, fontsize=18)  # Taille de la police définie à 16

fig.savefig(save_path+'/'+titre+'.png', dpi=300, bbox_inches='tight')  # Enregistre au format PNG avec une résolution de 300 DPI

plt.show()